# 🛡️ Do Support & Resistance Levels REALLY Hold?
### BacktestLAB — Video #10 — Scientific Backtest

---

## 🗺️ Road Map

| # | Section | Goal |
|---|---------|------|
| 1 | Setup & Data | Load price data for a demo asset|
| 2 | MA as S/R | Which Moving Average acts best as Support/Resistance? |
| 3 | Ichimoku as S/R | Do Kijun-sen and the Cloud really hold? |
| 4 | Multi-Asset Comparison | Test across stocks, indices & forex |
| 5 | Method Combination | Which combo gives the most robust signals? |
| 6 | Scientific Verdict | Final conclusions |

> ⚠️ Disclaimer — Educational purposes only. Not financial advice.


---
## ⚙️ SECTION 1 — Setup & Data

In [1]:
# ════════════════════════════════════════════════════════════════
# SECTION 1 — Libraries
# ════════════════════════════════════════════════════════════════
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import datetime
import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries loaded")


✅ Libraries loaded


In [2]:
# ════════════════════════════════════════════════════════════════
# SECTION 1 — Configuration
# ════════════════════════════════════════════════════════════════
'''
ASSETS = {
    'S&P 500':  '^GSPC',
    'NASDAQ':   '^IXIC',
    'Apple':    'AAPL',
    'EUR/USD':  'EURUSD=X',
    'Gold':     'GC=F',
}
'''

DEMO_SYMBOL  = 'GC=F'
DEMO_LABEL   = 'Gold'

YEARS        = 20
START_DATE   = (datetime.datetime.now() - datetime.timedelta(days=365*YEARS)).strftime('%Y-%m-%d')
END_DATE     = datetime.datetime.now().strftime('%Y-%m-%d')


# ── S/R Touch detection ──────────────────────────────────────────
TOUCH_TOL_PCT = 0.5       # How close (%) the candle must get to the level

# ── Outcome evaluation  ────────────────────────────────────
TARGET_R      = 2.0       # Take-Profit in R multiples (e.g. 2R)
# R definition for SUPPORT: entry = open[i+1], stop = bottom of zone (level * (1-tol))
# R definition for RESISTANCE: entry = open[i+1], stop = top of zone (level * (1+tol))

SR_DISPLAY_MODE = 'both'  # 'support' | 'resistance' | 'both'
PREV_BARS_ABOVE = 5   # number of preceding candles that must be clearly outside the zone

print(f"✅ Config — {DEMO_LABEL} | {YEARS}Y |  Target: {TARGET_R}R")


✅ Config — Gold | 20Y |  Target: 2.0R


In [3]:
# ════════════════════════════════════════════════════════════════
# SECTION 1 — Data Loading
# ════════════════════════════════════════════════════════════════
def getDataFromYahoo(symbol, start, end, frequency='1d', auto_adjust=True):
    try:
        data = yf.download(symbol, start=start, end=end,
                           interval=frequency, auto_adjust=auto_adjust,
                           progress=False)
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)
        df = data.reset_index()[['Date', 'Open', 'High', 'Low', 'Close', 'Volume']]
        df.columns = ['datetime', 'open', 'high', 'low', 'close', 'volume']
        df['datetime'] = pd.to_datetime(df['datetime']).dt.tz_localize(None)
        return df.reset_index(drop=True)
    except Exception as e:
        print(f"  ⚠️  Error fetching {symbol}: {e}")
        return None

df_main = getDataFromYahoo(DEMO_SYMBOL, START_DATE, END_DATE)
print(f"✅ {DEMO_LABEL} ({DEMO_SYMBOL})")
print(f"   Rows   : {len(df_main)}")
print(f"   Period : {df_main['datetime'].iloc[0].date()} → {df_main['datetime'].iloc[-1].date()}")


✅ Gold (GC=F)
   Rows   : 5026
   Period : 2006-05-16 → 2026-05-08


---
## 📐 SECTION 2 — Moving Averages as Dynamic Support & Resistance

**Concept:**  
A Moving Average acts as **support** when price approaches from above and bounces.  
It acts as **resistance** when price approaches from below and gets rejected.

**How we evaluate each touch (R-based system):**

| Outcome | Condition |
|---------|----------------------------------------|
| ✅ **WIN** | Price reaches **+2R** target (2× the risk distance) |
| ❌ **LOSS** | Price breaks through the zone (stop hit) |




In [4]:
MA_PERIODS   = [20, 50, 100, 200]

# ════════════════════════════════════════════════════════════════
# SECTION 2 — Compute Moving Averages
# ════════════════════════════════════════════════════════════════
def add_moving_averages(df, ma_periods):
    df = df.copy()
    for p in ma_periods:
        df[f'ma{p}'] = df['close'].rolling(p).mean()
    return df

df_main = add_moving_averages(df_main, MA_PERIODS)
print(f"✅ MAs added: {['ma'+str(p) for p in MA_PERIODS]}")


✅ MAs added: ['ma20', 'ma50', 'ma100', 'ma200']


In [17]:
# ════════════════════════════════════════════════════════════════
# SECTION 2 — S/R Touch Detection with R-based Outcome
# ════════════════════════════════════════════════════════════════
#
# For each touch we look forward candles and classify:
#
#   SUPPORT touch scenario:
#     Entry  = open of candle i+1
#     Stop   = zone bottom = ma_val * (1 - tol)      ← price breaks here → LOSS
#     Target = entry + TARGET_R * (entry - stop)      ← price reaches here → WIN
#
#
#   RESISTANCE touch scenario:
#     Entry  = open of candle i+1
#     Stop   = zone top  = ma_val * (1 + tol)         ← price breaks here → LOSS
#     Target = entry - TARGET_R * (stop - entry)       ← price reaches here → WIN
#
# ════════════════════════════════════════════════════════════════

# ════════════════════════════════════════════════════════════════
# SECTION 2 — S/R Touch Detection with R-based Outcome
# ════════════════════════════════════════════════════════════════

def detect_ma_sr_touches(df, ma_col, tol_pct=0.5, target_r=2.0, prev_bars=5):
    """
    Detect when price touches a Moving Average zone and evaluate
    the outcome using a dynamic MA-trailing stop and fixed R target.

    Parameters
    ----------
    ma_col   : MA column name (e.g. 'ma100')
    tol_pct  : tolerance zone width in % around the MA (e.g. 0.5 = ±0.5%)
    target_r : take-profit in R multiples (e.g. 2.0 = 2R)
    prev_bars: number of preceding candles checked for directional context

    Touch rules
    -----------
    Direction:
      price above MA  →  support only
      price below MA  →  resistance only

    Context filter:
      For support    : majority of prev_bars closes must be ABOVE zone upper band
      For resistance : majority of prev_bars closes must be BELOW zone lower band
      This rejects touches where price was already drifting inside the zone.

    Trade mechanics
    ---------------
      Entry       = open of candle i+1
      Initial stop= bottom of zone at entry candle  (ma[i] * (1-tol))
      Dynamic stop= ma[j] * (1-tol) updated on every evaluation candle
      Risk (R)    = entry - initial_stop
      Target      = entry + target_r * R  (fixed)

    Cooldown
    --------
      After a touch at i, scanning resumes from exit_idx+1.
      No overlapping evaluation windows.

    Outcomes
    --------
      WIN        : target reached
      LOSS       : dynamic stop hit
      INCOMPLETE : end of data reached before either level

    Trade details stored in df.attrs['trades_{ma_col}'] for plotting.
    """
    df    = df.copy()
    tol   = tol_pct / 100.0
    ma    = df[ma_col].values
    close = df['close'].values
    high  = df['high'].values
    low   = df['low'].values
    open_ = df['open'].values
    n     = len(df)

    sup_win        = np.zeros(n, dtype=bool)
    sup_loss       = np.zeros(n, dtype=bool)
    sup_incomplete = np.zeros(n, dtype=bool)
    res_win        = np.zeros(n, dtype=bool)
    res_loss       = np.zeros(n, dtype=bool)
    res_incomplete = np.zeros(n, dtype=bool)

    trades    = []
    min_count = prev_bars // 2 + 1

    i = max(prev_bars, 1)
    while i < n - 2:
        if np.isnan(ma[i]):
            i += 1
            continue

        ma_val     = ma[i]
        zone_upper = ma_val * (1 + tol)
        zone_lower = ma_val * (1 - tol)

        prev_close = close[i - prev_bars: i]
        prev_ma    = ma[i - prev_bars: i]
        prev_above = np.sum(prev_close > prev_ma * (1 + tol))
        prev_below = np.sum(prev_close < prev_ma * (1 - tol))

        # ── SUPPORT ──────────────────────────────────────────────
        if (close[i] > ma_val
                and low[i] <= zone_upper
                and low[i] >= zone_lower
                and prev_above >= min_count):

            entry_idx     = i + 1
            entry_px      = open_[entry_idx]
            initial_stop  = zone_lower
            risk          = entry_px - initial_stop

            if risk > 0:
                target_level = entry_px + target_r * risk

                outcome, exit_idx, n_bars = evaluate_sr_outcome(
                    high, low, ma, entry_idx, True, tol, target_level, initial_stop
                )

                # Dynamic stop value at exit for display
                exit_stop = ma[exit_idx] * (1 - tol) if not np.isnan(ma[exit_idx]) else initial_stop

                trades.append({
                    'type':          'support',
                    'touch_i':       i,
                    'entry_i':       entry_idx,
                    'exit_i':        exit_idx,
                    'n_bars':        n_bars,
                    'ma_val':        round(float(ma_val), 4),
                    'entry_px':      round(float(entry_px), 4),
                    'initial_stop':  round(float(initial_stop), 4),
                    'exit_stop':     round(float(exit_stop), 4),
                    'target':        round(float(target_level), 4),
                    'risk_pct':      round(float(risk / entry_px * 100), 3),
                    'outcome':       outcome,
                    'exit_px':       round(float(
                                       high[exit_idx] if outcome == 'win'
                                       else low[exit_idx]
                                     ), 4),
                    'touch_date':    df['datetime'].iloc[i],
                    'exit_date':     df['datetime'].iloc[exit_idx],
                })

                if   outcome == 'win':        sup_win[i]        = True
                elif outcome == 'loss':       sup_loss[i]       = True
                else:                         sup_incomplete[i] = True

                i = exit_idx + 1
                continue

        # ── RESISTANCE ───────────────────────────────────────────
        elif (close[i] < ma_val
                and high[i] >= zone_lower
                and high[i] <= zone_upper
                and prev_below >= min_count):

            entry_idx     = i + 1
            entry_px      = open_[entry_idx]
            initial_stop  = zone_upper
            risk          = initial_stop - entry_px

            if risk > 0:
                target_level = entry_px - target_r * risk

                outcome, exit_idx, n_bars = evaluate_sr_outcome(
                    high, low, ma, entry_idx, False, tol, target_level, initial_stop
                )

                exit_stop = ma[exit_idx] * (1 + tol) if not np.isnan(ma[exit_idx]) else initial_stop

                trades.append({
                    'type':          'resistance',
                    'touch_i':       i,
                    'entry_i':       entry_idx,
                    'exit_i':        exit_idx,
                    'n_bars':        n_bars,
                    'ma_val':        round(float(ma_val), 4),
                    'entry_px':      round(float(entry_px), 4),
                    'initial_stop':  round(float(initial_stop), 4),
                    'exit_stop':     round(float(exit_stop), 4),
                    'target':        round(float(target_level), 4),
                    'risk_pct':      round(float(risk / entry_px * 100), 3),
                    'outcome':       outcome,
                    'exit_px':       round(float(
                                       low[exit_idx] if outcome == 'win'
                                       else high[exit_idx]
                                     ), 4),
                    'touch_date':    df['datetime'].iloc[i],
                    'exit_date':     df['datetime'].iloc[exit_idx],
                })

                if   outcome == 'win':        res_win[i]        = True
                elif outcome == 'loss':       res_loss[i]       = True
                else:                         res_incomplete[i] = True

                i = exit_idx + 1
                continue

        i += 1

    df[f'sr_sup_win_{ma_col}']        = sup_win
    df[f'sr_sup_loss_{ma_col}']       = sup_loss
    df[f'sr_sup_incomplete_{ma_col}'] = sup_incomplete
    df[f'sr_res_win_{ma_col}']        = res_win
    df[f'sr_res_loss_{ma_col}']       = res_loss
    df[f'sr_res_incomplete_{ma_col}'] = res_incomplete

    df.attrs[f'trades_{ma_col}'] = trades

    return df


def evaluate_sr_outcome(highs, lows, ma_vals, entry_idx, is_support,
                         tol, target_level, stop_level):
    """
    Scan forward from entry_idx until WIN or LOSS — no time limit.

    The stop loss is dynamic: it follows the MA zone on every candle.
      Support    : dynamic_stop = ma[j] * (1 - tol)  — moves UP with the MA
      Resistance : dynamic_stop = ma[j] * (1 + tol)  — moves DOWN with the MA

    The target is fixed at entry + target_r * initial_risk.

    Parameters
    ----------
    highs, lows : price arrays
    ma_vals     : MA values array (used for dynamic stop)
    entry_idx   : first candle to evaluate (open[entry_idx] = entry price)
    is_support  : True for support, False for resistance
    tol         : tolerance as decimal (e.g. 0.005 for 0.5%)
    target_level: fixed take-profit price level

    Returns
    -------
    outcome  : 'win', 'loss', or 'incomplete' (end of data reached)
    exit_idx : candle index where outcome was triggered
    n_bars   : number of candles from entry to exit
    """
    n = len(highs)
    for j in range(entry_idx, n):
        if np.isnan(ma_vals[j]):
            continue

        #dynamic_stop = ma_vals[j] * (1 - tol) if is_support else ma_vals[j] * (1 + tol)
        dynamic_stop = stop_level

        if is_support:
            if lows[j] < dynamic_stop:
                return 'loss', j, j - entry_idx
            if highs[j] >= target_level:
                return 'win', j, j - entry_idx
        else:
            if highs[j] > dynamic_stop:
                return 'loss', j, j - entry_idx
            if lows[j] <= target_level:
                return 'win', j, j - entry_idx

    return 'incomplete', n - 1, n - 1 - entry_idx





# ── Apply — add prev_bars=PREV_BARS_ABOVE ────────────────────────
for p in MA_PERIODS:
    df_main = detect_ma_sr_touches(
        df_main, f'ma{p}',
        tol_pct=TOUCH_TOL_PCT,
        target_r=TARGET_R,
        prev_bars=PREV_BARS_ABOVE
    )
print("✅ S/R touches evaluated")


✅ S/R touches evaluated


In [18]:
# ════════════════════════════════════════════════════════════════
# SECTION 2 — Score each MA
# ════════════════════════════════════════════════════════════════
def score_ma_sr(df, ma_periods):
    rows = []
    for p in ma_periods:
        mc = f'ma{p}'
        sw = df[f'sr_sup_win_{mc}'].sum()
        sl = df[f'sr_sup_loss_{mc}'].sum()
        si = df[f'sr_sup_incomplete_{mc}'].sum()
        rw = df[f'sr_res_win_{mc}'].sum()
        rl = df[f'sr_res_loss_{mc}'].sum()
        ri = df[f'sr_res_incomplete_{mc}'].sum()

        wins    = sw + rw
        losses  = sl + rl
        decided = wins + losses
        win_rate = round(wins / decided * 100, 1) if decided > 0 else 0

        # Average bars to exit (decided trades only)
        trades = df.attrs.get(f'trades_{mc}', [])
        decided_trades = [t for t in trades if t['outcome'] in ('win', 'loss')]
        avg_bars = round(np.mean([t['n_bars'] for t in decided_trades]), 1) if decided_trades else 0

        rows.append({
            'MA':              f'MA{p}',
            'WIN  ✅':          int(wins),
            'LOSS ❌':          int(losses),
            'Incomplete ⏳':   int(si + ri),
            'Total Decided':   int(decided),
            'Win Rate %':      win_rate,
            'Avg Bars':        avg_bars,
            'Expectancy':      round((win_rate/100 * TARGET_R) - ((1 - win_rate/100) * 1), 2)
        })
    return pd.DataFrame(rows)

score_df = score_ma_sr(df_main, MA_PERIODS)
display(score_df.style
    .background_gradient(cmap='RdYlGn', subset=['Win Rate %', 'Expectancy'])
    .format({'Win Rate %': '{:.1f}%', 'Expectancy': '{:+.2f}R'})
    .set_caption(f'📊 MA S/R — {DEMO_LABEL} | Target: {TARGET_R}R'))


,MA,WIN ✅,LOSS ❌,Incomplete ⏳,Total Decided,Win Rate %,Avg Bars,Expectancy
0,MA20,75,186,0,261,28.7%,4.000000,-0.14R
1,MA50,68,104,0,172,39.5%,5.400000,+0.19R
2,MA100,47,78,0,125,37.6%,4.600000,+0.13R
3,MA200,23,49,0,72,31.9%,3.800000,-0.04R


In [19]:
# =========================================================
# V2 - MA Support / Resistance detection with candlestick confirmation
# =========================================================

def detect_ma_sr_touches_v2(df, ma_col, tol_pct=0.5):
    """
    Detect MA support and resistance touches using candlestick confirmations.

    Support conditions:
    1. Bullish engulfing pattern near the MA
       -> Entry on the next candle

    2. Hammer or Morning Star type candle near the MA
       -> Next candle must be bullish
       -> Entry on the following candle

    Resistance logic is the exact opposite.
    """

    df = df.copy()

    # =========================================================
    # Candle components
    # =========================================================

    body = (df['close'] - df['open']).abs()
    csize = df['high'] - df['low']

    low_sh  = np.minimum(df['open'], df['close']) - df['low']
    high_sh = df['high'] - np.maximum(df['open'], df['close'])

    prev_open  = df['open'].shift(1)
    prev_close = df['close'].shift(1)

    # =========================================================
    # Bullish engulfing
    # =========================================================

    _bull_eng = (
        (df['close'] > df['open']) &
        (prev_close < prev_open) &
        (df['open']  <= prev_close) &
        (df['close'] > prev_open)
    )


    # =========================================================
    # Bearish engulfing
    # =========================================================

    _bear_eng = (
        (df['close'] < df['open']) &
        (prev_close > prev_open) &
        (df['open']  >= prev_close) &
        (df['close'] < prev_open)
    )

    # =========================================================
    # Hammer / Star
    # =========================================================

    _hammer = (
        (body > 0) &
        (low_sh  >= 2 * body) &
        (high_sh <= 0.3 * csize)
    )

    _star = (
        (body > 0) &
        (high_sh >= 2 * body) &
        (low_sh  <= 0.3 * csize)
    )

    # =========================================================
    # Confirmation candles
    # =========================================================

    bullish_confirm = df['close'].shift(-1) > df['open'].shift(-1)
    bearish_confirm = df['close'].shift(-1) < df['open'].shift(-1)

    # =========================================================
    # MA support detection
    # =========================================================

    tol   = tol_pct / 100.0
    zone_upper = ma_col * (1 + tol)
    zone_lower = ma_col * (1 - tol)

    ma_support_touch = (
        (df['low'] <= zone_upper) &
        (df['low'] >= zone_lower) &
        (df['close'] >= df[ma_col])
    )



    # Bullish engulfing -> enter on candle 3
    support_engulfing = (
        ma_support_touch &
        _bull_eng
    ).shift(1)

    # Hammer -> next candle must be bullish -> enter on candle 3
    support_hammer = (
        ma_support_touch &
        _hammer &
        bullish_confirm
    ).shift(2)

    # Star -> next candle must be bullish -> enter on candle 3
    support_star = (
        ma_support_touch &
        _star &
        bullish_confirm
    ).shift(2)

    df['ma_support_signal_v2'] = (
        support_engulfing |
        support_hammer |
        support_star
    )

    # =========================================================
    # MA resistance detection
    # =========================================================

    ma_resistance_touch = (
        (df['high'] >= zone_lower) &
        (df['high'] <= zone_upper) &
        (df['close'] <= df[ma_col])
    )


    # Bearish engulfing -> enter on candle 3
    resistance_engulfing = (
        ma_resistance_touch &
        _bear_eng
    ).shift(1)

    # Shooting star -> next candle must be bearish -> enter on candle 3
    resistance_star = (
        ma_resistance_touch &
        _star &
        bearish_confirm
    ).shift(2)

    # Inverted hammer style rejection -> bearish confirmation
    resistance_hammer = (
        ma_resistance_touch &
        _hammer &
        bearish_confirm
    ).shift(2)

    df['ma_resistance_signal_v2'] = (
        resistance_engulfing |
        resistance_star |
        resistance_hammer
    )

    return df


In [20]:
# ════════════════════════════════════════════════════════════════
# SECTION 2 — Plot S/R Touches with WIN / LOSS / NEUTRAL colours
# ════════════════════════════════════════════════════════════════
def plot_ma_sr(df, symbol, ma_col, mode='both', last_n_bars=500, show_debug=True):
    """
    Enhanced plot:
      - Shaded tolerance zone around the MA (semi-transparent)
      - 2R target line drawn for every WIN trade
      - Hover annotations: entry / stop / target / risk% / outcome
      - show_debug=True → print trade table below the chart
    """
    df_plot  = df.tail(last_n_bars).copy()
    all_trades = df.attrs.get(f'trades_{ma_col}', [])

    # Filter trades that fall inside the plot window
    first_dt  = df_plot['datetime'].iloc[0]
    trades_in = [t for t in all_trades if t['touch_date'] >= first_dt]

    fig = go.Figure()

    # ── Candlesticks ─────────────────────────────────────────────
    fig.add_trace(go.Candlestick(
        x=df_plot['datetime'], open=df_plot['open'], high=df_plot['high'],
        low=df_plot['low'], close=df_plot['close'], name='Price',
        increasing_line_color='#26a69a', decreasing_line_color='#ef5350'
    ))

    # ── MA line ──────────────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=df_plot['datetime'], y=df_plot[ma_col],
        line=dict(color='royalblue', width=2), name=ma_col.upper()
    ))

    # ── Tolerance zone (shaded band around MA) ───────────────────
    tol  = TOUCH_TOL_PCT / 100.0
    zone_upper = df_plot[ma_col] * (1 + tol)
    zone_lower = df_plot[ma_col] * (1 - tol)

    fig.add_trace(go.Scatter(
        x=df_plot['datetime'], y=zone_upper,
        line=dict(width=0), showlegend=False, hoverinfo='skip'
    ))
    fig.add_trace(go.Scatter(
        x=df_plot['datetime'], y=zone_lower,
        line=dict(width=0), fill='tonexty',
        fillcolor='rgba(100, 149, 237, 0.15)',   # cornflower blue, transparent
        name='Tolerance Zone', hoverinfo='skip'
    ))

    # ── WIN / LOSS / NEUTRAL markers (with hover info) ───────────
    marker_cfg = {
        'sup_win':     ('triangle-up',   '#00e676', 'low',  0.997),
        'sup_loss':    ('x',             '#ff1744', 'low',  0.997),
        'sup_neutral': ('circle',        '#888888', 'low',  0.997),
        'res_win':     ('triangle-down', '#00e676', 'high', 1.003),
        'res_loss':    ('x',             '#ff1744', 'high', 1.003),
        'res_neutral': ('circle',        '#888888', 'high', 1.003),
    }

    # Build lookup: touch_date → trade info
    trade_lookup = {t['touch_date']: t for t in trades_in}

    for key, (sym, color, price_col, offset) in marker_cfg.items():
        sr_type, outcome = key.split('_', 1)
        if sr_type == 'sup' and mode not in ('support', 'both'):   continue
        if sr_type == 'res' and mode not in ('resistance', 'both'): continue

        col_name = f'sr_{key}_{ma_col}'
        if col_name not in df_plot.columns: continue
        pts = df_plot[df_plot[col_name]].copy()
        if pts.empty: continue

        y_vals     = pts[price_col] * offset
        hover_txts = []
        for _, row in pts.iterrows():
            t = trade_lookup.get(row['datetime'], {})
            if t:
                hover_txts.append(
                    f"<b>{sr_type.upper()} {outcome.upper()}</b><br>"
                    f"Pattern: {t.get('pattern', 'MA')}<br>"
                    f"Date   : {t['touch_date'].strftime('%Y-%m-%d')}<br>"
                    f"MA level: {t['ma_val']}<br>"
                    f"Entry   : {t['entry_px']}<br>"
                    f"Init Stop    : {t['initial_stop']}<br>"
                    f"Exit Stop    : {t['exit_stop']}<br>"
                    f"Target  : {t['target']}  ({TARGET_R}R)<br>"
                    f"Risk    : {t['risk_pct']}%<br>"
                    f"Exit    : {t['exit_px']}  ({t['exit_date'].strftime('%Y-%m-%d') if hasattr(t['exit_date'],'strftime') else ''})<br>"
                    f"Outcome : {t['outcome'].upper()}"
                )
            else:
                hover_txts.append(outcome.upper())

        lbl_map = {
            'sup_win': '✅ Support WIN', 'sup_loss': '❌ Support LOSS',
            'sup_neutral': '⚪ Support NEUTRAL',
            'res_win': '✅ Resistance WIN', 'res_loss': '❌ Resistance LOSS',
            'res_neutral': '⚪ Resistance NEUTRAL',
        }
        fig.add_trace(go.Scatter(
            x=pts['datetime'], y=y_vals, mode='markers',
            marker=dict(symbol=sym, size=11, color=color,
                        line=dict(color='white', width=0.5)),
            name=lbl_map[key],
            text=hover_txts, hovertemplate='%{text}<extra></extra>'
        ))

    # ── Entry / initial_stop / Target lines for ALL trades ───────────────
    outcome_colors = {
        'win':     {'target': '#69ff47', 'entry': '#29b6f6', 'initial_stop': '#ff1744'},
        'loss':    {'target': '#444444', 'entry': '#888888', 'initial_stop': '#ff1744'},
        'neutral': {'target': '#555555', 'entry': '#555555', 'initial_stop': '#555555'},
    }

    for t in trades_in:
        colors   = outcome_colors.get(t['outcome'], outcome_colors['neutral'])
        x0       = t['touch_date']
        x1       = t['exit_date']
        is_win   = t['outcome'] == 'win'
        is_loss  = t['outcome'] == 'loss'

        # Target (2R) line
        fig.add_shape(
            type='line', x0=x0, x1=x1,
            y0=t['target'], y1=t['target'],
            line=dict(color=colors['target'], width=1.5 if is_win else 0.8,
                      dash='dot'),
            opacity=0.9 if is_win else 0.4
        )

        # Entry line
        fig.add_shape(
            type='line', x0=x0, x1=x1,
            y0=t['entry_px'], y1=t['entry_px'],
            line=dict(color=colors['entry'], width=1.0, dash='dash'),
            opacity=0.7 if is_win else 0.35
        )

        # initial_stop line
        fig.add_shape(
            type='line', x0=x0, x1=x1,
            y0=t['initial_stop'], y1=t['initial_stop'],
            line=dict(color=colors['initial_stop'], width=1.0, dash='dash'),
            opacity=0.7 if is_loss else 0.35
        )

        # Annotation: target label for WIN, dimmed for others
        fig.add_annotation(
            x=x1, y=t['target'],
            text=f"{TARGET_R}R={t['target']:.0f}",
            showarrow=False,
            font=dict(size=9, color=colors['target']),
            xanchor='left', yanchor='middle',
            opacity=0.9 if is_win else 0.4
        )

        # Annotation: risk % for all trades
        fig.add_annotation(
            x=x0, y=t['entry_px'],
            text=f"R={t['risk_pct']}%",
            showarrow=False,
            font=dict(size=8, color=colors['entry']),
            xanchor='right', yanchor='middle',
            opacity=0.8 if is_win else 0.4
        )

    title_text = f'BackTestLAB — {symbol}  |  {ma_col.upper()} S/R  |   Target {TARGET_R}R  |  Prev check: {PREV_BARS_ABOVE} bars  |  Mode: {mode}'

    fig.update_layout(
        height=680, template='plotly_dark',
        title=dict(
            text=title_text,
            font=dict(size=14, color='gold')
        ),
        xaxis_rangeslider_visible=False,
        legend=dict(orientation='h', y=1.05),
        hoverlabel=dict(bgcolor='#1e1e2e', font_size=11)
    )
    fig.update_xaxes(rangebreaks=[dict(bounds=['sat', 'mon'])])
    fig.show()

    # ── Debug table (show_debug=True) ────────────────────────────
    if show_debug and trades_in:
        debug_rows = []
        for t in trades_in:
            debug_rows.append({
                'Type':       t['type'],
                'Date':       t['touch_date'].strftime('%Y-%m-%d'),
                'MA Level':   t['ma_val'],
                'Entry':      t['entry_px'],
                'Init Stop':       t['initial_stop'],
                'Exit Stop':       t['exit_stop'],
                f'Target {TARGET_R}R': t['target'],
                'Risk %':     f"{t['risk_pct']}%",
                'Bars to Exit': t['n_bars'],
                'Outcome':    t['outcome'].upper(),
                'Exit Date':  t['exit_date'].strftime('%Y-%m-%d') if hasattr(t['exit_date'], 'strftime') else '',
                'Exit Price': t['exit_px'],
            })
        debug_df = pd.DataFrame(debug_rows)

        def color_row(row):
            if row['Outcome'] == 'WIN':   bg = '#1a3a1a; color: #b5f5a0'
            elif row['Outcome'] == 'LOSS': bg = '#3a1a1a; color: #f5a0a0'
            else:                          bg = '#2a2a2a; color: #aaaaaa'
            return [f'background-color: {bg}'] * len(row)

        display(debug_df.style
            .apply(color_row, axis=1)
            .set_caption(f'📋 Trade Details — {ma_col.upper()} | Last {last_n_bars} bars'))



best_ma_period = MA_PERIODS[score_df['Win Rate %'].idxmax()]
best_ma_col    = f'ma{best_ma_period}'
best_rate      = score_df.loc[score_df['MA'] == f'MA{best_ma_period}', 'Win Rate %'].values[0]
print(f"🏆 Best MA: MA{best_ma_period}  |  Win Rate: {best_rate}%")
plot_ma_sr(df_main, DEMO_LABEL, best_ma_col, mode=SR_DISPLAY_MODE, last_n_bars=500)


🏆 Best MA: MA50  |  Win Rate: 39.5%


,Type,Date,MA Level,Entry,Init Stop,Exit Stop,Target 2.0R,Risk %,Bars to Exit,Outcome,Exit Date,Exit Price
0,support,2024-05-30,2318.856000,2344.100100,2307.261700,2326.867200,2417.776900,1.572%,5,LOSS,2024-06-07,2285.399900
1,resistance,2024-06-12,2343.286000,2309.399900,2355.002400,2348.713100,2218.194900,1.975%,13,LOSS,2024-07-03,2361.600100
2,support,2024-07-26,2361.892000,2377.300000,2350.082500,2350.637700,2431.735100,1.145%,2,WIN,2024-07-31,2447.600100
3,support,2024-08-05,2362.802000,2414.500000,2350.988000,2434.723200,2541.524000,2.63%,26,WIN,2024-09-12,2557.000000
4,resistance,2024-11-20,2657.362000,2659.300000,2670.648800,2673.078900,2636.602500,0.427%,0,LOSS,2024-11-21,2672.100100
5,resistance,2024-11-29,2666.964000,2649.000000,2680.298800,2681.384200,2586.402400,1.182%,6,LOSS,2024-12-10,2698.200000
6,resistance,2025-01-02,2662.858000,2658.700000,2676.172300,2672.650800,2623.755300,0.657%,1,WIN,2025-01-06,2617.300000
7,resistance,2025-01-07,2657.790000,2655.500000,2671.078900,2669.543300,2624.342100,0.587%,0,LOSS,2025-01-08,2676.899900
8,support,2025-04-07,2939.756000,2994.000000,2925.057200,2943.100500,3131.885500,2.303%,2,WIN,2025-04-10,3167.000000
9,support,2025-06-24,3312.412000,3321.600100,3295.849900,3301.593100,3373.100400,0.775%,2,LOSS,2025-06-27,3253.800000


In [23]:
def detect_ma_sr_touches_v2(df, ma_col, tol_pct=0.5, target_r=2.0, prev_bars=5):
    """
    v2 of detect_ma_sr_touches — identical logic but requires a candlestick
    pattern confirmation before entering any trade.

    Pattern rules (support)
    -----------------------
    Bullish Engulfing : candle i-1 bearish + candle i bullish engulfs it
                        → entry = open[i+1]

    Hammer            : candle i has long lower shadow (≥ 2× body)
                        + candle i+1 confirms (closes above hammer HIGH)
                        → entry = open[i+2]

    Pattern rules (resistance)
    --------------------------
    Bearish Engulfing : candle i-1 bullish + candle i bearish engulfs it
                        → entry = open[i+1]

    Shooting Star     : candle i has long upper shadow (≥ 2× body)
                        + candle i+1 confirms (closes below star LOW)
                        → entry = open[i+2]

    If no pattern is detected at a valid S/R touch, the touch is skipped
    and scanning resumes from i+1.  Cooldown and outcome logic are
    identical to v1 (dynamic stop, no time limit, WIN / LOSS / INCOMPLETE).

    The 'pattern' key is added to every trade dict in df.attrs.
    """
    df    = df.copy()
    tol   = tol_pct / 100.0
    ma    = df[ma_col].values
    close = df['close'].values
    high  = df['high'].values
    low   = df['low'].values
    open_ = df['open'].values
    n     = len(df)

    sup_win        = np.zeros(n, dtype=bool)
    sup_loss       = np.zeros(n, dtype=bool)
    sup_incomplete = np.zeros(n, dtype=bool)
    res_win        = np.zeros(n, dtype=bool)
    res_loss       = np.zeros(n, dtype=bool)
    res_incomplete = np.zeros(n, dtype=bool)

    trades    = []
    min_count = prev_bars // 2 + 1

    # ── Candle pattern helpers ────────────────────────────────────────

    def is_bull_engulfing(i):
        """Candle i-1 bearish, candle i bullish and fully engulfs i-1."""
        if i < 1:
            return False
        return (
            close[i]   > open_[i]    and   # i is bullish
            close[i-1] < open_[i-1]  and   # i-1 is bearish
            open_[i]  <= close[i-1]  and   # opens at or below prev close
            close[i]   > open_[i-1]        # closes above prev open
        )

    def is_bear_engulfing(i):
        """Candle i-1 bullish, candle i bearish and fully engulfs i-1."""
        if i < 1:
            return False
        return (
            close[i]   < open_[i]    and
            close[i-1] > open_[i-1]  and
            open_[i]  >= close[i-1]  and
            close[i]   < open_[i-1]
        )

    def is_hammer(i):
        """Long lower shadow (≥ 2× body) and tiny upper shadow (≤ 30% candle)."""
        body = abs(close[i] - open_[i])
        if body <= 0:
            return False
        low_sh  = min(close[i], open_[i]) - low[i]
        high_sh = high[i] - max(close[i], open_[i])
        csize   = high[i] - low[i]
        return low_sh >= 2 * body and high_sh <= 0.3 * csize

    def is_shooting_star(i):
        """Long upper shadow (≥ 2× body) and tiny lower shadow (≤ 30% candle)."""
        body = abs(close[i] - open_[i])
        if body <= 0:
            return False
        high_sh = high[i] - max(close[i], open_[i])
        low_sh  = min(close[i], open_[i]) - low[i]
        csize   = high[i] - low[i]
        return high_sh >= 2 * body and low_sh <= 0.3 * csize

    # ── Pattern + entry index resolution ─────────────────────────────

    def find_support_pattern(i):
        """
        Return (entry_idx, pattern_name) for a bullish pattern at touch i,
        or (None, None) if no pattern is found.

        Engulfing checked first (no extra lookahead needed).
        Hammer checked second (needs i+1 as confirmation bar).
        """
        # Bullish Engulfing: confirmation = candle i itself, entry at i+1
        if is_bull_engulfing(i) and i + 1 < n:
            return i + 1, 'bull_engulfing'

        # Hammer: confirmation = close[i+1] > high[i], entry at i+2
        if i + 2 < n and is_hammer(i) and close[i + 1] > open_[i + 1] and close[i+1] > ma[i+1]:
            return i + 2, 'hammer'

        # Shooting Star: confirmation = close[i+1] < low[i], entry at i+2
        if i + 2 < n and is_shooting_star(i) and close[i + 1] > open_[i + 1] and close[i+1] > ma[i+1]:
            return i + 2, 'shooting_star'

        return None, None

    def find_resistance_pattern(i):
        """
        Return (entry_idx, pattern_name) for a bearish pattern at touch i,
        or (None, None) if no pattern is found.
        """
        # Bearish Engulfing: entry at i+1
        if is_bear_engulfing(i) and i + 1 < n:

            return i + 1, 'bear_engulfing'

        # Shooting Star: confirmation = close[i+1] < low[i], entry at i+2
        if i + 2 < n and is_shooting_star(i) and close[i + 1] < open_[i + 1] and close[i+1] < ma[i+1]:
            return i + 2, 'shooting_star'

        # hammer: confirmation = close[i+1] < low[i], entry at i+2
        if i + 2 < n and is_hammer(i) and close[i + 1] < open_[i + 1] and close[i+1]  < ma[i+1]:
            return i + 2, 'hammer'

        return None, None

    # ── Main scanning loop ────────────────────────────────────────────

    i = max(prev_bars, 1)
    while i < n - 2:
        if np.isnan(ma[i]):
            i += 1
            continue

        ma_val     = ma[i]
        zone_upper = ma_val * (1 + tol)
        zone_lower = ma_val * (1 - tol)

        prev_close = close[i - prev_bars: i]
        prev_ma    = ma[i - prev_bars: i]
        prev_above = np.sum(prev_close > prev_ma * (1 + tol))
        prev_below = np.sum(prev_close < prev_ma * (1 - tol))

        # ── SUPPORT ──────────────────────────────────────────────────
        if (close[i] > ma_val
                and low[i] <= zone_upper
                and prev_above >= min_count):

            entry_idx, pattern_name = find_support_pattern(i)

            if entry_idx is not None:
                entry_px     = open_[entry_idx]
                initial_stop = zone_lower
                risk         = entry_px - initial_stop

                if risk > 0:
                    target_level = entry_px + target_r * risk

                    outcome, exit_idx, n_bars = evaluate_sr_outcome(
                        high, low, ma, entry_idx, True, tol, target_level, initial_stop
                    )

                    exit_stop = (ma[exit_idx] * (1 - tol)
                                 if not np.isnan(ma[exit_idx]) else initial_stop)

                    trades.append({
                        'type':         'support',
                        'pattern':      pattern_name,
                        'touch_i':      i,
                        'entry_i':      entry_idx,
                        'exit_i':       exit_idx,
                        'n_bars':       n_bars,
                        'ma_val':       round(float(ma_val), 4),
                        'entry_px':     round(float(entry_px), 4),
                        'initial_stop': round(float(initial_stop), 4),
                        'exit_stop':    round(float(exit_stop), 4),
                        'target':       round(float(target_level), 4),
                        'risk_pct':     round(float(risk / entry_px * 100), 3),
                        'outcome':      outcome,
                        'exit_px':      round(float(
                                          high[exit_idx] if outcome == 'win'
                                          else low[exit_idx]
                                        ), 4),
                        'touch_date':   df['datetime'].iloc[i],
                        'exit_date':    df['datetime'].iloc[exit_idx],
                    })

                    if   outcome == 'win':        sup_win[i]        = True
                    elif outcome == 'loss':       sup_loss[i]       = True
                    else:                         sup_incomplete[i] = True

                    i = exit_idx + 1
                    continue

            # S/R touch valid but no pattern → skip, keep scanning
            i += 1
            continue

        # ── RESISTANCE ───────────────────────────────────────────────
        elif (close[i] < ma_val
                and high[i] >= zone_lower
                and prev_below >= min_count):

            entry_idx, pattern_name = find_resistance_pattern(i)

            if entry_idx is not None:
                entry_px     = open_[entry_idx]
                initial_stop = zone_upper
                risk         = initial_stop - entry_px

                if risk > 0:
                    target_level = entry_px - target_r * risk

                    outcome, exit_idx, n_bars = evaluate_sr_outcome(
                        high, low, ma, entry_idx, False, tol, target_level, initial_stop
                    )

                    exit_stop = (ma[exit_idx] * (1 + tol)
                                 if not np.isnan(ma[exit_idx]) else initial_stop)

                    trades.append({
                        'type':         'resistance',
                        'pattern':      pattern_name,
                        'touch_i':      i,
                        'entry_i':      entry_idx,
                        'exit_i':       exit_idx,
                        'n_bars':       n_bars,
                        'ma_val':       round(float(ma_val), 4),
                        'entry_px':     round(float(entry_px), 4),
                        'initial_stop': round(float(initial_stop), 4),
                        'exit_stop':    round(float(exit_stop), 4),
                        'target':       round(float(target_level), 4),
                        'risk_pct':     round(float(risk / entry_px * 100), 3),
                        'outcome':      outcome,
                        'exit_px':      round(float(
                                          low[exit_idx] if outcome == 'win'
                                          else high[exit_idx]
                                        ), 4),
                        'touch_date':   df['datetime'].iloc[i],
                        'exit_date':    df['datetime'].iloc[exit_idx],
                    })

                    if   outcome == 'win':        res_win[i]        = True
                    elif outcome == 'loss':       res_loss[i]       = True
                    else:                         res_incomplete[i] = True

                    i = exit_idx + 1
                    continue

            i += 1
            continue

        i += 1

    df[f'sr_sup_win_{ma_col}']        = sup_win
    df[f'sr_sup_loss_{ma_col}']       = sup_loss
    df[f'sr_sup_incomplete_{ma_col}'] = sup_incomplete
    df[f'sr_res_win_{ma_col}']        = res_win
    df[f'sr_res_loss_{ma_col}']       = res_loss
    df[f'sr_res_incomplete_{ma_col}'] = res_incomplete

    df.attrs[f'trades_{ma_col}'] = trades

    return df


df_main_v2 = getDataFromYahoo(DEMO_SYMBOL, START_DATE, END_DATE)
df_main_v2 = add_moving_averages(df_main_v2, MA_PERIODS)
print(f"✅ MAs added: {['ma'+str(p) for p in MA_PERIODS]}")

# ── Apply — add prev_bars=PREV_BARS_ABOVE ────────────────────────
for p in MA_PERIODS:
    df_main_v2 = detect_ma_sr_touches_v2(
        df_main_v2, f'ma{p}',
        tol_pct=TOUCH_TOL_PCT,
        target_r=TARGET_R,
        prev_bars=PREV_BARS_ABOVE
    )
print("✅ S/R touches evaluated")

score_df_v2 = score_ma_sr(df_main_v2, MA_PERIODS)
display(score_df_v2.style
    .background_gradient(cmap='RdYlGn', subset=['Win Rate %', 'Expectancy'])
    .format({'Win Rate %': '{:.1f}%', 'Expectancy': '{:+.2f}R'})
    .set_caption(f'📊 MA S/R — {DEMO_LABEL} | Target: {TARGET_R}R'))

print ("\n old V1 values to compare with:")
display(score_df.style
    .background_gradient(cmap='RdYlGn', subset=['Win Rate %', 'Expectancy'])
    .format({'Win Rate %': '{:.1f}%', 'Expectancy': '{:+.2f}R'})
    .set_caption(f'📊 MA S/R — {DEMO_LABEL} | Target: {TARGET_R}R'))

✅ MAs added: ['ma20', 'ma50', 'ma100', 'ma200']
✅ S/R touches evaluated


,MA,WIN ✅,LOSS ❌,Incomplete ⏳,Total Decided,Win Rate %,Avg Bars,Expectancy
0,MA20,22,38,0,60,36.7%,8.700000,+0.10R
1,MA50,16,28,0,44,36.4%,9.300000,+0.09R
2,MA100,8,21,0,29,27.6%,12.800000,-0.17R
3,MA200,6,11,0,17,35.3%,6.500000,+0.06R



 old V1 values to compare with:


,MA,WIN ✅,LOSS ❌,Incomplete ⏳,Total Decided,Win Rate %,Avg Bars,Expectancy
0,MA20,75,186,0,261,28.7%,4.000000,-0.14R
1,MA50,68,104,0,172,39.5%,5.400000,+0.19R
2,MA100,47,78,0,125,37.6%,4.600000,+0.13R
3,MA200,23,49,0,72,31.9%,3.800000,-0.04R


In [24]:
best_ma_period_v2 = MA_PERIODS[score_df_v2['Win Rate %'].idxmax()]
best_ma_col_v2    = 'ma20' #f'ma{best_ma_period_v2}'
best_rate_v2      = score_df_v2.loc[score_df_v2['MA'] == f'MA{best_ma_period_v2}', 'Win Rate %'].values[0]
print(f"🏆 Best MA: MA{best_ma_period_v2}  |  Win Rate: {best_rate_v2}%")
plot_ma_sr(df_main_v2, DEMO_LABEL, best_ma_col_v2, mode=SR_DISPLAY_MODE, last_n_bars=2000)

🏆 Best MA: MA20  |  Win Rate: 36.7%


,Type,Date,MA Level,Entry,Init Stop,Exit Stop,Target 2.0R,Risk %,Bars to Exit,Outcome,Exit Date,Exit Price
0,resistance,2018-07-09,1268.115000,1253.200000,1274.455600,1237.974100,1210.688700,1.696%,16,WIN,2018-08-02,1205.800000
1,support,2019-07-09,1380.730000,1424.000000,1373.826300,1440.814700,1524.347300,3.523%,23,WIN,2019-08-13,1531.400000
2,resistance,2019-10-21,1495.200000,1483.400000,1502.676000,1498.992700,1444.848100,1.299%,3,LOSS,2019-10-25,1514.300000
3,support,2020-08-12,1930.095000,1938.700000,1920.444500,1943.120600,1975.210800,0.942%,1,WIN,2020-08-17,1985.000000
4,resistance,2021-01-29,1866.580000,1859.600000,1875.912900,1856.566700,1826.974100,0.877%,2,WIN,2021-02-04,1782.800000
5,resistance,2021-02-10,1841.320000,1825.000000,1850.526600,1842.592100,1773.946800,1.399%,2,WIN,2021-02-17,1768.800000
6,resistance,2021-03-16,1740.825000,1745.800000,1749.529100,1745.232700,1738.341900,0.214%,0,LOSS,2021-03-18,1750.300000
7,resistance,2021-10-04,1768.220000,1759.300000,1777.061100,1770.061300,1723.778000,1.01%,2,LOSS,2021-10-08,1780.000000
8,support,2021-12-29,1788.140000,1825.100000,1779.199300,1809.173700,1916.901300,2.515%,19,LOSS,2022-01-28,1778.800000
9,resistance,2022-06-29,1836.070000,1795.500000,1845.250400,1764.192100,1695.999300,2.771%,13,WIN,2022-07-21,1679.800000


---
## ☁️ SECTION 3 — Ichimoku as Dynamic Support & Resistance

**Concept:**  
- **Kijun-sen** (Base Line, 26) — institutional equilibrium, the most respected level
- **Senkou Span A** — upper/lower cloud edge
- **Senkou Span B** — the thicker, stronger S/R inside the cloud

Same **R-based outcome** logic applies here.


In [25]:
# ════════════════════════════════════════════════════════════════
# SECTION 3 — Compute Ichimoku
# ════════════════════════════════════════════════════════════════
def compute_ichimoku(df, tenkan=9, kijun=26, senkou_b=52):
    df = df.copy()
    df['tenkan_sen'] = (df['high'].rolling(tenkan).max() +
                        df['low'].rolling(tenkan).min()) / 2
    df['kijun_sen']  = (df['high'].rolling(kijun).max() +
                        df['low'].rolling(kijun).min())  / 2
    df['senkou_a']   = ((df['tenkan_sen'] + df['kijun_sen']) / 2).shift(kijun)
    df['senkou_b']   = ((df['high'].rolling(senkou_b).max() +
                         df['low'].rolling(senkou_b).min()) / 2).shift(kijun)
    df['chikou']     = df['close'].shift(-kijun)
    return df

df_main = compute_ichimoku(df_main)
print("✅ Ichimoku computed")


✅ Ichimoku computed


In [26]:
# ════════════════════════════════════════════════════════════════
# SECTION 3 — Detect Ichimoku S/R with Dynamic Stop
# ════════════════════════════════════════════════════════════════
def detect_ichimoku_sr(df, tol_pct=0.5, target_r=2.0, prev_bars=5):
    """
    Detect S/R touches on Kijun-sen, Senkou A and Senkou B
    using the same rules as detect_ma_sr_touches:

    Direction:
      price above level  →  support only
      price below level  →  resistance only

    Context filter:
      For support    : majority of prev_bars closes above the zone upper band
      For resistance : majority of prev_bars closes below the zone lower band

    Stop loss:
      Dynamic — follows the Ichimoku level on every evaluation candle
      Support    : dynamic_stop = level[j] * (1 - tol)
      Resistance : dynamic_stop = level[j] * (1 + tol)

    Target: fixed at entry + target_r * initial_risk

    Cooldown:
      After a touch at i, scanning resumes from exit_idx + 1.

    Outcomes: 'win', 'loss', 'incomplete' (end of data)

    Trade details stored in df.attrs['ichi_trades_{level}'] for plotting.
    """
    df    = df.copy()
    tol   = tol_pct / 100.0
    n     = len(df)
    high  = df['high'].values
    low   = df['low'].values
    open_ = df['open'].values
    close = df['close'].values

    min_count = prev_bars // 2 + 1

    for level in ['kijun_sen', 'senkou_a', 'senkou_b']:
        lvl = df[level].values

        sup_win        = np.zeros(n, dtype=bool)
        sup_loss       = np.zeros(n, dtype=bool)
        sup_incomplete = np.zeros(n, dtype=bool)
        res_win        = np.zeros(n, dtype=bool)
        res_loss       = np.zeros(n, dtype=bool)
        res_incomplete = np.zeros(n, dtype=bool)

        trades = []

        i = max(prev_bars, 1)
        while i < n - 2:
            if np.isnan(lvl[i]):
                i += 1
                continue

            lv         = lvl[i]
            zone_upper = lv * (1 + tol)
            zone_lower = lv * (1 - tol)

            prev_close = close[i - prev_bars: i]
            prev_lvl   = lvl[i - prev_bars: i]
            valid_prev = ~np.isnan(prev_lvl)
            prev_above = np.sum(prev_close[valid_prev] > prev_lvl[valid_prev] * (1 + tol))
            prev_below = np.sum(prev_close[valid_prev] < prev_lvl[valid_prev] * (1 - tol))

            # ── SUPPORT ──────────────────────────────────────────
            if (close[i] > lv
                    and low[i] <= zone_upper
                    and prev_above >= min_count):

                entry_idx    = i + 1
                entry_px     = open_[entry_idx]
                initial_stop = zone_lower
                risk         = entry_px - initial_stop

                if risk > 0:
                    target_level = entry_px + target_r * risk

                    outcome, exit_idx, n_bars = evaluate_sr_outcome(
                        high, low, lvl, entry_idx, True, tol, target_level, initial_stop
                    )

                    exit_stop = lvl[exit_idx] * (1 - tol) if not np.isnan(lvl[exit_idx]) else initial_stop

                    trades.append({
                        'type':         'support',
                        'touch_i':      i,
                        'entry_i':      entry_idx,
                        'exit_i':       exit_idx,
                        'n_bars':       n_bars,
                        'level_val':    round(float(lv), 4),
                        'entry_px':     round(float(entry_px), 4),
                        'initial_stop': round(float(initial_stop), 4),
                        'exit_stop':    round(float(exit_stop), 4),
                        'target':       round(float(target_level), 4),
                        'risk_pct':     round(float(risk / entry_px * 100), 3),
                        'outcome':      outcome,
                        'exit_px':      round(float(
                                          high[exit_idx] if outcome == 'win'
                                          else low[exit_idx]
                                        ), 4),
                        'touch_date':   df['datetime'].iloc[i],
                        'exit_date':    df['datetime'].iloc[exit_idx],
                    })

                    if   outcome == 'win':        sup_win[i]        = True
                    elif outcome == 'loss':       sup_loss[i]       = True
                    else:                         sup_incomplete[i] = True

                    i = exit_idx + 1
                    continue

            # ── RESISTANCE ───────────────────────────────────────
            elif (close[i] < lv
                    and high[i] >= zone_lower
                    and prev_below >= min_count):

                entry_idx    = i + 1
                entry_px     = open_[entry_idx]
                initial_stop = zone_upper
                risk         = initial_stop - entry_px

                if risk > 0:
                    target_level = entry_px - target_r * risk

                    outcome, exit_idx, n_bars = evaluate_sr_outcome(
                        high, low, lvl, entry_idx, False, tol, target_level, initial_stop
                    )

                    exit_stop = lvl[exit_idx] * (1 + tol) if not np.isnan(lvl[exit_idx]) else initial_stop

                    trades.append({
                        'type':         'resistance',
                        'touch_i':      i,
                        'entry_i':      entry_idx,
                        'exit_i':       exit_idx,
                        'n_bars':       n_bars,
                        'level_val':    round(float(lv), 4),
                        'entry_px':     round(float(entry_px), 4),
                        'initial_stop': round(float(initial_stop), 4),
                        'exit_stop':    round(float(exit_stop), 4),
                        'target':       round(float(target_level), 4),
                        'risk_pct':     round(float(risk / entry_px * 100), 3),
                        'outcome':      outcome,
                        'exit_px':      round(float(
                                          low[exit_idx] if outcome == 'win'
                                          else high[exit_idx]
                                        ), 4),
                        'touch_date':   df['datetime'].iloc[i],
                        'exit_date':    df['datetime'].iloc[exit_idx],
                    })

                    if   outcome == 'win':        res_win[i]        = True
                    elif outcome == 'loss':       res_loss[i]       = True
                    else:                         res_incomplete[i] = True

                    i = exit_idx + 1
                    continue

            i += 1

        df[f'ichi_sup_win_{level}']        = sup_win
        df[f'ichi_sup_loss_{level}']       = sup_loss
        df[f'ichi_sup_incomplete_{level}'] = sup_incomplete
        df[f'ichi_res_win_{level}']        = res_win
        df[f'ichi_res_loss_{level}']       = res_loss
        df[f'ichi_res_incomplete_{level}'] = res_incomplete
        df.attrs[f'ichi_trades_{level}']   = trades

    return df

df_main = detect_ichimoku_sr(df_main, TOUCH_TOL_PCT, TARGET_R, PREV_BARS_ABOVE)
print("✅ Ichimoku S/R evaluated")


✅ Ichimoku S/R evaluated


In [27]:
# ════════════════════════════════════════════════════════════════
# SECTION 3 — Score Ichimoku Levels
# ════════════════════════════════════════════════════════════════
def score_ichimoku_sr(df):
    rows = []
    for level in ['kijun_sen', 'senkou_a', 'senkou_b']:
        wins       = df[f'ichi_sup_win_{level}'].sum()        + df[f'ichi_res_win_{level}'].sum()
        losses     = df[f'ichi_sup_loss_{level}'].sum()       + df[f'ichi_res_loss_{level}'].sum()
        incomplete = df[f'ichi_sup_incomplete_{level}'].sum() + df[f'ichi_res_incomplete_{level}'].sum()
        decided    = wins + losses
        win_rate   = round(wins / decided * 100, 1) if decided > 0 else 0

        trades         = df.attrs.get(f'ichi_trades_{level}', [])
        decided_trades = [t for t in trades if t['outcome'] in ('win', 'loss')]
        avg_bars       = round(np.mean([t['n_bars'] for t in decided_trades]), 1) if decided_trades else 0

        rows.append({
            'Ichimoku Level':  level.replace('_', ' ').title(),
            'WIN  ✅':          int(wins),
            'LOSS ❌':          int(losses),
            'Incomplete ⏳':   int(incomplete),
            'Total Decided':   int(decided),
            'Win Rate %':      win_rate,
            'Avg Bars':        avg_bars,
            'Expectancy':      round((win_rate / 100 * TARGET_R) - (1 - win_rate / 100), 2)
        })
    return pd.DataFrame(rows)

ichi_score_df = score_ichimoku_sr(df_main)
display(ichi_score_df.style
    .background_gradient(cmap='RdYlGn', subset=['Win Rate %', 'Expectancy'])
    .format({'Win Rate %': '{:.1f}%', 'Expectancy': '{:+.2f}R'})
    .set_caption(f'📊 Ichimoku S/R — {DEMO_LABEL} | Target {TARGET_R}R | Dynamic Stop'))


,Ichimoku Level,WIN ✅,LOSS ❌,Incomplete ⏳,Total Decided,Win Rate %,Avg Bars,Expectancy
0,Kijun Sen,82,169,0,251,32.7%,4.200000,-0.02R
1,Senkou A,55,87,0,142,38.7%,4.200000,+0.16R
2,Senkou B,38,74,0,112,33.9%,5.400000,+0.02R


In [28]:
# ════════════════════════════════════════════════════════════════
# SECTION 3 — Plot Ichimoku S/R
# ════════════════════════════════════════════════════════════════
def plot_ichimoku_sr(df, symbol, level='kijun_sen', mode='both',
                     last_n_bars=500, show_debug=True):
    df_plot    = df.tail(last_n_bars).copy()
    first_dt   = df_plot['datetime'].iloc[0]
    all_trades = df.attrs.get(f'ichi_trades_{level}', [])
    trades_in  = [t for t in all_trades if t['touch_date'] >= first_dt]
    lbl        = level.replace('_', ' ').title()
    tol        = TOUCH_TOL_PCT / 100.0

    fig = go.Figure()

    # ── Candlesticks ─────────────────────────────────────────────
    fig.add_trace(go.Candlestick(
        x=df_plot['datetime'], open=df_plot['open'], high=df_plot['high'],
        low=df_plot['low'], close=df_plot['close'], name='Price',
        increasing_line_color='#26a69a', decreasing_line_color='#ef5350'
    ))

    # ── Ichimoku cloud ───────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=df_plot['datetime'], y=df_plot['senkou_a'],
        line=dict(width=0), showlegend=False, hoverinfo='skip'
    ))
    fig.add_trace(go.Scatter(
        x=df_plot['datetime'], y=df_plot['senkou_b'],
        line=dict(width=0), fill='tonexty',
        fillcolor='rgba(100,180,100,0.12)', name='Cloud', hoverinfo='skip'
    ))

    # ── Kijun and Tenkan ─────────────────────────────────────────
    fig.add_trace(go.Scatter(
        x=df_plot['datetime'], y=df_plot['kijun_sen'],
        line=dict(color='#ef5350', width=1.5), name='Kijun-sen'
    ))
    fig.add_trace(go.Scatter(
        x=df_plot['datetime'], y=df_plot['tenkan_sen'],
        line=dict(color='#29b6f6', width=1), name='Tenkan-sen'
    ))

    # ── Tolerance zone around the active level ───────────────────
    zone_upper = df_plot[level] * (1 + tol)
    zone_lower = df_plot[level] * (1 - tol)
    fig.add_trace(go.Scatter(
        x=df_plot['datetime'], y=zone_upper,
        line=dict(width=0), showlegend=False, hoverinfo='skip'
    ))
    fig.add_trace(go.Scatter(
        x=df_plot['datetime'], y=zone_lower,
        line=dict(width=0), fill='tonexty',
        fillcolor='rgba(239,83,80,0.12)',
        name=f'Zone ±{TOUCH_TOL_PCT}% ({lbl})', hoverinfo='skip'
    ))

    # ── WIN / LOSS / INCOMPLETE markers with hover ───────────────
    trade_lookup = {t['touch_date']: t for t in trades_in}

    marker_cfg = [
        (f'ichi_sup_win_{level}',        'triangle-up',   '#00e676', 'low',  0.997, f'✅ Support WIN ({lbl})'),
        (f'ichi_sup_loss_{level}',       'x',             '#ff1744', 'low',  0.997, f'❌ Support LOSS ({lbl})'),
        (f'ichi_sup_incomplete_{level}', 'circle',        '#888888', 'low',  0.997, f'⏳ Support INCOMPLETE ({lbl})'),
        (f'ichi_res_win_{level}',        'triangle-down', '#00e676', 'high', 1.003, f'✅ Resistance WIN ({lbl})'),
        (f'ichi_res_loss_{level}',       'x',             '#ff1744', 'high', 1.003, f'❌ Resistance LOSS ({lbl})'),
        (f'ichi_res_incomplete_{level}', 'circle',        '#888888', 'high', 1.003, f'⏳ Resistance INCOMPLETE ({lbl})'),
    ]

    for col, sym, color, price_col, offset, name in marker_cfg:
        sr_type = 'sup' if 'sup' in col else 'res'
        if sr_type == 'sup' and mode not in ('support', 'both'):   continue
        if sr_type == 'res' and mode not in ('resistance', 'both'): continue
        if col not in df_plot.columns: continue

        pts        = df_plot[df_plot[col]].copy()
        if pts.empty: continue
        hover_txts = []
        for _, row in pts.iterrows():
            t = trade_lookup.get(row['datetime'], {})
            if t:
                hover_txts.append(
                    f"<b>{t['type'].upper()} {t['outcome'].upper()}</b><br>"
                    f"Level ({lbl}) : {t['level_val']}<br>"
                    f"Entry         : {t['entry_px']}<br>"
                    f"Init Stop     : {t['initial_stop']}<br>"
                    f"Exit Stop     : {t['exit_stop']}<br>"
                    f"Target {TARGET_R}R   : {t['target']}<br>"
                    f"Risk          : {t['risk_pct']}%<br>"
                    f"Outcome       : {t['outcome'].upper()}<br>"
                    f"Bars to exit  : {t['n_bars']}<br>"
                    f"Exit date     : {t['exit_date'].strftime('%Y-%m-%d')}"
                )
            else:
                hover_txts.append(name)

        fig.add_trace(go.Scatter(
            x=pts['datetime'], y=pts[price_col] * offset, mode='markers',
            marker=dict(symbol=sym, size=11, color=color,
                        line=dict(color='white', width=0.5)),
            name=name,
            text=hover_txts, hovertemplate='%{text}<extra></extra>'
        ))

    # ── Entry / Stop / Target lines for all trades ───────────────
    outcome_colors = {
        'win':        {'target': '#69ff47', 'entry': '#29b6f6', 'stop': '#ff4444'},
        'loss':       {'target': '#444444', 'entry': '#888888', 'stop': '#ff1744'},
        'incomplete': {'target': '#333333', 'entry': '#444444', 'stop': '#444444'},
    }

    for t in trades_in:
        c      = outcome_colors.get(t['outcome'], outcome_colors['incomplete'])
        is_win  = t['outcome'] == 'win'
        is_loss = t['outcome'] == 'loss'
        x0, x1  = t['touch_date'], t['exit_date']

        fig.add_shape(type='line', x0=x0, x1=x1,
            y0=t['target'], y1=t['target'],
            line=dict(color=c['target'], width=1.5 if is_win else 0.8, dash='dot'),
            opacity=0.9 if is_win else 0.35)

        fig.add_shape(type='line', x0=x0, x1=x1,
            y0=t['entry_px'], y1=t['entry_px'],
            line=dict(color=c['entry'], width=1.0, dash='dash'),
            opacity=0.7 if is_win else 0.3)

        fig.add_shape(type='line', x0=x0, x1=x1,
            y0=t['initial_stop'], y1=t['initial_stop'],
            line=dict(color=c['stop'], width=1.0, dash='dash'),
            opacity=0.7 if is_loss else 0.3)

        if is_win:
            fig.add_annotation(
                x=x1, y=t['target'],
                text=f"{TARGET_R}R={t['target']:.0f}",
                showarrow=False,
                font=dict(size=9, color=c['target']),
                xanchor='left', yanchor='middle'
            )
        fig.add_annotation(
            x=x0, y=t['entry_px'],
            text=f"R={t['risk_pct']}%",
            showarrow=False,
            font=dict(size=8, color=c['entry']),
            xanchor='right', yanchor='middle',
            opacity=0.8 if is_win else 0.4
        )

    title_text = f'BackTestLAB — {symbol}  |  Ichimoku — {lbl}  |  Target {TARGET_R}R  |  Dynamic Stop  |  Mode: {mode}'
    fig.update_layout(
        height=680, template='plotly_dark',
        title=dict(text=title_text, font=dict(size=14, color='gold')),
        xaxis_rangeslider_visible=False,
        legend=dict(orientation='h', y=1.05),
        hoverlabel=dict(bgcolor='#1e1e2e', font_size=11)
    )
    fig.update_xaxes(rangebreaks=[dict(bounds=['sat', 'mon'])])
    fig.show()

    # ── Debug table ──────────────────────────────────────────────
    if show_debug and trades_in:
        debug_rows = [{
            'Type':            t['type'],
            'Touch Date':      t['touch_date'].strftime('%Y-%m-%d'),
            f'{lbl} Level':    t['level_val'],
            'Entry':           t['entry_px'],
            'Init Stop':       t['initial_stop'],
            'Exit Stop':       t['exit_stop'],
            f'Target {TARGET_R}R': t['target'],
            'Risk %':          f"{t['risk_pct']}%",
            'Outcome':         t['outcome'].upper(),
            'Exit Date':       t['exit_date'].strftime('%Y-%m-%d'),
            'Exit Price':      t['exit_px'],
            'Bars to Exit':    t['n_bars'],
        } for t in trades_in]

        def color_row(row):
            if row['Outcome'] == 'WIN':        return [f'background-color: #1a3a1a; color: #b5f5a0'] * len(row)
            elif row['Outcome'] == 'LOSS':     return [f'background-color: #3a1a1a; color: #f5a0a0'] * len(row)
            else:                              return [f'background-color: #2a2a2a; color: #aaaaaa'] * len(row)

        display(pd.DataFrame(debug_rows).style
            .apply(color_row, axis=1)
            .set_caption(f'📋 Ichimoku Trade Details — {lbl} | Last {last_n_bars} bars'))


best_ichi_idx   = ichi_score_df['Win Rate %'].idxmax()
best_ichi_level = ['kijun_sen', 'senkou_a', 'senkou_b'][best_ichi_idx]
best_ichi_rate  = ichi_score_df.loc[best_ichi_idx, 'Win Rate %']
print(f"🏆 Best Ichimoku level: {best_ichi_level}  |  Win Rate: {best_ichi_rate}%")
plot_ichimoku_sr(df_main, DEMO_LABEL, level=best_ichi_level, mode=SR_DISPLAY_MODE, last_n_bars=1000)

🏆 Best Ichimoku level: senkou_a  |  Win Rate: 38.7%


,Type,Touch Date,Senkou A Level,Entry,Init Stop,Exit Stop,Target 2.0R,Risk %,Outcome,Exit Date,Exit Price,Bars to Exit
0,resistance,2022-08-10,1810.475000,1794.300000,1819.527400,1779.654000,1743.845400,1.406%,WIN,2022-08-22,1726.500000,7
1,resistance,2022-08-24,1756.775000,1761.600000,1765.558900,1746.187500,1753.682200,0.225%,WIN,2022-08-26,1736.100000,1
2,resistance,2022-08-29,1737.200000,1736.400000,1745.886000,1745.760400,1717.428000,0.546%,WIN,2022-08-31,1708.500000,1
3,support,2023-02-15,1831.850000,1838.600000,1822.690800,1823.362400,1870.418400,0.865%,LOSS,2023-02-17,1818.400000,1
4,support,2023-05-19,1964.675000,1980.500000,1954.851700,1981.418200,2031.796700,1.295%,LOSS,2023-05-25,1943.100000,3
5,resistance,2023-07-18,1979.250000,1977.000000,1989.146200,1969.749800,1952.707500,0.614%,WIN,2023-07-27,1945.400000,6
6,resistance,2023-08-29,1945.375000,1936.000000,1955.101900,1928.394000,1897.796300,0.987%,WIN,2023-09-27,1871.600000,19
7,resistance,2023-10-16,1924.875000,1914.300000,1934.499400,1933.444100,1873.901400,1.055%,LOSS,2023-10-18,1957.900000,1
8,support,2024-01-10,2012.175000,2025.100000,2002.114100,2046.391600,2071.071800,1.135%,LOSS,2024-02-13,1990.000000,22
9,resistance,2024-02-20,2037.575000,2028.200000,2047.762900,2047.561900,1989.074100,0.965%,LOSS,2024-02-29,2049.800000,6


---
## 🌍 SECTION 4 — Multi-Asset Comparison

Same pipeline on all 5 assets — which market respects S/R the most?


In [29]:
'''ASSETS = {
    'S&P 500':  '^GSPC',
    'NASDAQ':   '^IXIC',
    'Apple':    'AAPL',
    'EUR/USD':  'EURUSD=X',
    'Gold':     'GC=F',
}
'''

ASSETS = {
    # TECH
    "Apple": "AAPL",
    "Microsoft": "MSFT",
    "Nvidia": "NVDA",
    "Alphabet": "GOOGL",
    "Meta": "META",
    "Amazon": "AMZN",
    "Oracle": "ORCL",
    "Salesforce": "CRM",
    "Adobe": "ADBE",
    "Intel": "INTC",
    "AMD": "AMD",
    "Cisco": "CSCO",
    "IBM": "IBM",
    "Qualcomm": "QCOM",
    "ServiceNow": "NOW",

    # CONSUMER
    "Walmart": "WMT",
    "Costco": "COST",
    "Home Depot": "HD",
    "Lowe's": "LOW",
    "Nike": "NKE",
    "Starbucks": "SBUX",
    "McDonald's": "MCD",
    "Disney": "DIS",
    "Target": "TGT",

    # HEALTHCARE
    "Johnson & Johnson": "JNJ",
    "UnitedHealth": "UNH",
    "Abbott": "ABT",
    "Medtronic": "MDT",
    "Thermo Fisher": "TMO",
    "Danaher": "DHR",
    "Intuitive Surgical": "ISRG",
    "Stryker": "SYK",

    # FINANCE
    "Visa": "V",
    "Mastercard": "MA",
    "JPMorgan": "JPM",
    "Bank of America": "BAC",
    "Morgan Stanley": "MS",
    "Goldman Sachs": "GS",
    "American Express": "AXP",

    # INDUSTRIAL / ENERGY
    "Caterpillar": "CAT",
    "John Deere": "DE",
    "General Electric": "GE",
    "Honeywell": "HON",
    "Union Pacific": "UNP",
    "NextEra Energy": "NEE",

    # EXTRA (to have exactly 50)
    "ExxonMobil": "XOM",
    "Chevron": "CVX",
    "Coca-Cola": "KO",
    "PepsiCo": "PEP",
    "Berkshire Hathaway": "BRK-B"
}



# ════════════════════════════════════════════════════════════════
# SECTION 4 — Multi-Asset Pipeline (MA + Ichimoku )
# ════════════════════════════════════════════════════════════════
def run_full_sr_analysis(symbol, label, start, end,
                          ma_periods, tol_pct, target_r, prev_bars=5):
    df = getDataFromYahoo(symbol, start, end)
    if df is None or len(df) < 300:
        return None

    # ── MA ───────────────────────────────────────────────────────
    df = add_moving_averages(df, ma_periods)
    for p in ma_periods:
        df = detect_ma_sr_touches(df, f'ma{p}', tol_pct, target_r, prev_bars)

    # ── Ichimoku ─────────────────────────────────────────────────
    df = compute_ichimoku(df)
    df = detect_ichimoku_sr(df, tol_pct, target_r, prev_bars)

    def wr(w, l):
        return round(w / (w + l) * 100, 1) if (w + l) > 0 else 0

    def exp(win_rate):
        return round((win_rate / 100 * target_r) - (1 - win_rate / 100), 2)

    # ── Best MA (per period) ──────────────────────────────────────
    ma_detail = {}
    best_ma_wr, best_ma_lbl = 0, ''
    for p in ma_periods:
        mc = f'ma{p}'
        w  = df[f'sr_sup_win_{mc}'].sum()  + df[f'sr_res_win_{mc}'].sum()
        l  = df[f'sr_sup_loss_{mc}'].sum() + df[f'sr_res_loss_{mc}'].sum()
        rate = wr(w, l)
        ma_detail[f'MA{p}'] = rate
        if rate > best_ma_wr:
            best_ma_wr, best_ma_lbl = rate, f'MA{p}'

    # ── Best Ichimoku (per level) ─────────────────────────────────
    ichi_detail = {}
    best_ichi_wr, best_ichi_lbl = 0, ''
    for lv in ['kijun_sen', 'senkou_a', 'senkou_b']:
        w    = df[f'ichi_sup_win_{lv}'].sum()  + df[f'ichi_res_win_{lv}'].sum()
        l    = df[f'ichi_sup_loss_{lv}'].sum() + df[f'ichi_res_loss_{lv}'].sum()
        rate = wr(w, l)
        ichi_detail[lv.replace('_', ' ').title()] = rate
        if rate > best_ichi_wr:
            best_ichi_wr, best_ichi_lbl = rate, lv.replace('_', ' ').title()


    return {
        'Asset':              label,
        # ── Summary ─────────────────────────────────────────────
        'Best MA':            best_ma_lbl,
        'MA Win Rate %':      best_ma_wr,
        'MA Expectancy':      exp(best_ma_wr),
        'Best Ichimoku':      best_ichi_lbl,
        'Ichi Win Rate %':    best_ichi_wr,
        'Ichi Expectancy':    exp(best_ichi_wr),
        # ── Per-config detail (for championship tables) ──────────
        '_ma_detail':         ma_detail,
        '_ichi_detail':       ichi_detail,
    }


print(f"⏳ Running multi-asset analysis | Target {TARGET_R}R ...")
multi_results = []
for label, symbol in ASSETS.items():
    print(f"  {label} ({symbol})...", end=" ")
    r = run_full_sr_analysis(
        symbol, label, START_DATE, END_DATE,
        MA_PERIODS, TOUCH_TOL_PCT, TARGET_R
    )
    if r:
        multi_results.append(r)
        print(f"✅  MA: {r['MA Win Rate %']}%  "
              f"Ichi: {r['Ichi Win Rate %']}%  ")
    else:
        print("⚠️  skipped")

multi_df = pd.DataFrame(multi_results)
print(f"\n✅ Done — {len(multi_df)} assets analysed")

⏳ Running multi-asset analysis | Target 2.0R ...
  Apple (AAPL)... ✅  MA: 36.5%  Ichi: 37.9%  
  Microsoft (MSFT)... ✅  MA: 42.4%  Ichi: 36.7%  
  Nvidia (NVDA)... ✅  MA: 38.6%  Ichi: 34.7%  
  Alphabet (GOOGL)... ✅  MA: 37.9%  Ichi: 34.8%  
  Meta (META)... ✅  MA: 37.2%  Ichi: 38.9%  
  Amazon (AMZN)... ✅  MA: 36.3%  Ichi: 36.1%  
  Oracle (ORCL)... ✅  MA: 31.3%  Ichi: 32.5%  
  Salesforce (CRM)... ✅  MA: 34.7%  Ichi: 33.6%  
  Adobe (ADBE)... ✅  MA: 38.1%  Ichi: 42.0%  
  Intel (INTC)... ✅  MA: 32.9%  Ichi: 33.6%  
  AMD (AMD)... ✅  MA: 38.0%  Ichi: 32.0%  
  Cisco (CSCO)... ✅  MA: 38.4%  Ichi: 33.5%  
  IBM (IBM)... ✅  MA: 39.5%  Ichi: 37.1%  
  Qualcomm (QCOM)... ✅  MA: 35.0%  Ichi: 35.9%  
  ServiceNow (NOW)... ✅  MA: 49.0%  Ichi: 38.6%  
  Walmart (WMT)... ✅  MA: 36.2%  Ichi: 37.0%  
  Costco (COST)... ✅  MA: 35.2%  Ichi: 39.1%  
  Home Depot (HD)... ✅  MA: 45.5%  Ichi: 34.6%  
  Lowe's (LOW)... ✅  MA: 34.6%  Ichi: 36.5%  
  Nike (NKE)... ✅  MA: 36.4%  Ichi: 34.9%  
  Starbucks (

In [30]:
# ════════════════════════════════════════════════════════════════
# SECTION 4 — Table 1 : Asset-level summary
# ════════════════════════════════════════════════════════════════
display_cols = [
    'Asset',
    'Best MA', 'MA Win Rate %', 'MA Expectancy',
    'Best Ichimoku', 'Ichi Win Rate %', 'Ichi Expectancy',
]

display(multi_df[display_cols].style
    .background_gradient(cmap='RdYlGn',
        subset=['MA Win Rate %', 'Ichi Win Rate %'])
    .format({
        'MA Win Rate %':   '{:.1f}%',
        'Ichi Win Rate %': '{:.1f}%',
        'MA Expectancy':   '{:+.2f}R',
        'Ichi Expectancy': '{:+.2f}R',
    })
    .set_caption(f'📊 Multi-Asset S/R Comparison | Target {TARGET_R}R | {len(multi_df)} assets'))

,Asset,Best MA,MA Win Rate %,MA Expectancy,Best Ichimoku,Ichi Win Rate %,Ichi Expectancy
0,Apple,MA20,36.5%,+0.09R,Senkou B,37.9%,+0.14R
1,Microsoft,MA50,42.4%,+0.27R,Senkou B,36.7%,+0.10R
2,Nvidia,MA100,38.6%,+0.16R,Senkou B,34.7%,+0.04R
3,Alphabet,MA100,37.9%,+0.14R,Kijun Sen,34.8%,+0.04R
4,Meta,MA50,37.2%,+0.12R,Senkou B,38.9%,+0.17R
5,Amazon,MA50,36.3%,+0.09R,Kijun Sen,36.1%,+0.08R
6,Oracle,MA200,31.3%,-0.06R,Senkou A,32.5%,-0.03R
7,Salesforce,MA100,34.7%,+0.04R,Kijun Sen,33.6%,+0.01R
8,Adobe,MA100,38.1%,+0.14R,Senkou B,42.0%,+0.26R
9,Intel,MA200,32.9%,-0.01R,Kijun Sen,33.6%,+0.01R


In [31]:
# ════════════════════════════════════════════════════════════════
# SECTION 4 — Table 2 : Method championship
#   Which method wins the most assets? Which config dominates?
# ════════════════════════════════════════════════════════════════
def build_championship_tables(multi_results, ma_periods, target_r):
    n = len(multi_results)

    # ── 1. Method wins (which method had the highest win rate per asset) ──
    method_wins = {'MA': 0, 'Ichimoku': 0}
    for r in multi_results:
        rates = {
            'MA':           r['MA Win Rate %'],
            'Ichimoku':     r['Ichi Win Rate %'],
        }
        winner = max(rates, key=rates.get)
        method_wins[winner] += 1

    champ_method = pd.DataFrame([{
        'Method':           m,
        'Best on N assets': cnt,
        'Best on % assets': f'{round(cnt/n*100, 1)}%',
        'Avg Win Rate %':   round(np.mean([
            r['MA Win Rate %']   if m == 'MA' else
            r['Ichi Win Rate %']
            for r in multi_results
        ]), 1),
        'Avg Expectancy':   round(np.mean([
            r['MA Expectancy']   if m == 'MA' else
            r['Ichi Expectancy']
            for r in multi_results
        ]), 2),
    } for m, cnt in method_wins.items()])
    champ_method = champ_method.sort_values('Best on N assets', ascending=False)

    # ── 2. MA period championship ─────────────────────────────────
    ma_labels = [f'MA{p}' for p in ma_periods]
    ma_count  = {lbl: 0 for lbl in ma_labels}
    ma_rates  = {lbl: [] for lbl in ma_labels}
    for r in multi_results:
        ma_count[r['Best MA']] += 1
        for lbl in ma_labels:
            ma_rates[lbl].append(r['_ma_detail'].get(lbl, 0))

    champ_ma = pd.DataFrame([{
        'MA Period':         lbl,
        'Best on N assets':  ma_count[lbl],
        'Best on % assets':  f'{round(ma_count[lbl]/n*100, 1)}%',
        'Avg Win Rate %':    round(np.mean(ma_rates[lbl]), 1),
        'Avg Expectancy':    round((np.mean(ma_rates[lbl])/100*target_r)
                                   - (1 - np.mean(ma_rates[lbl])/100), 2),
    } for lbl in ma_labels])
    champ_ma = champ_ma.sort_values('Best on N assets', ascending=False)

    # ── 3. Ichimoku level championship ───────────────────────────
    ichi_labels = ['Kijun Sen', 'Senkou A', 'Senkou B']
    ichi_count  = {lbl: 0 for lbl in ichi_labels}
    ichi_rates  = {lbl: [] for lbl in ichi_labels}
    for r in multi_results:
        ichi_count[r['Best Ichimoku']] += 1
        for lbl in ichi_labels:
            ichi_rates[lbl].append(r['_ichi_detail'].get(lbl, 0))

    champ_ichi = pd.DataFrame([{
        'Ichimoku Level':    lbl,
        'Best on N assets':  ichi_count[lbl],
        'Best on % assets':  f'{round(ichi_count[lbl]/n*100, 1)}%',
        'Avg Win Rate %':    round(np.mean(ichi_rates[lbl]), 1),
        'Avg Expectancy':    round((np.mean(ichi_rates[lbl])/100*target_r)
                                   - (1 - np.mean(ichi_rates[lbl])/100), 2),
    } for lbl in ichi_labels])
    champ_ichi = champ_ichi.sort_values('Best on N assets', ascending=False)


    return champ_method, champ_ma, champ_ichi


champ_method, champ_ma, champ_ichi = build_championship_tables(
    multi_results, MA_PERIODS, TARGET_R
)

In [32]:
# ════════════════════════════════════════════════════════════════
# SECTION 4 — Display championship tables
# ════════════════════════════════════════════════════════════════
grad_cols = ['Avg Win Rate %', 'Avg Expectancy']

print("=" * 60)
print("🏆  WHICH METHOD WINS THE MOST ASSETS?")
print("=" * 60)
display(champ_method.style
    .background_gradient(cmap='RdYlGn', subset=grad_cols)
    .format({'Avg Win Rate %': '{:.1f}%', 'Avg Expectancy': '{:+.2f}R'})
    .hide(axis='index')
    .set_caption(f'Method Championship — {len(multi_results)} assets | Target {TARGET_R}R'))

print("\n" + "=" * 60)
print("📈  BEST MA PERIOD ACROSS ALL ASSETS")
print("=" * 60)
display(champ_ma.style
    .background_gradient(cmap='RdYlGn', subset=grad_cols)
    .format({'Avg Win Rate %': '{:.1f}%', 'Avg Expectancy': '{:+.2f}R'})
    .hide(axis='index')
    .set_caption('MA Period Championship'))

print("\n" + "=" * 60)
print("☁️   BEST ICHIMOKU LEVEL ACROSS ALL ASSETS")
print("=" * 60)
display(champ_ichi.style
    .background_gradient(cmap='RdYlGn', subset=grad_cols)
    .format({'Avg Win Rate %': '{:.1f}%', 'Avg Expectancy': '{:+.2f}R'})
    .hide(axis='index')
    .set_caption('Ichimoku Level Championship'))

🏆  WHICH METHOD WINS THE MOST ASSETS?


Method,Best on N assets,Best on % assets,Avg Win Rate %,Avg Expectancy
MA,31,62.0%,37.1%,+0.11R
Ichimoku,19,38.0%,35.3%,+0.06R



📈  BEST MA PERIOD ACROSS ALL ASSETS


MA Period,Best on N assets,Best on % assets,Avg Win Rate %,Avg Expectancy
MA200,16,32.0%,33.4%,+0.00R
MA50,14,28.0%,33.4%,+0.00R
MA100,11,22.0%,32.6%,-0.02R
MA20,9,18.0%,32.7%,-0.02R



☁️   BEST ICHIMOKU LEVEL ACROSS ALL ASSETS


Ichimoku Level,Best on N assets,Best on % assets,Avg Win Rate %,Avg Expectancy
Senkou B,22,44.0%,32.7%,-0.02R
Kijun Sen,15,30.0%,32.5%,-0.03R
Senkou A,13,26.0%,31.8%,-0.04R


In [33]:
# ════════════════════════════════════════════════════════════════
# SECTION 4 — Visual summary (bar chart)
# ════════════════════════════════════════════════════════════════
def plot_multi_asset_championship(multi_df, champ_method, champ_ma,
                                   champ_ichi):
    from plotly.subplots import make_subplots

    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            '🏆 Method — Avg Win Rate % across assets',
            '📈 Best MA Period — Times ranked #1',
            '☁️  Best Ichimoku Level — Times ranked #1',
        )
    )

    # Panel 1: method avg win rate
    fig.add_trace(go.Bar(
        x=champ_method['Method'],
        y=champ_method['Avg Win Rate %'],
        marker_color=['#2962ff', '#e91e63', '#f77f00'],
        text=champ_method['Avg Win Rate %'].apply(lambda x: f'{x:.1f}%'),
        textposition='outside', showlegend=False
    ), row=1, col=1)

    # Panel 2: MA period wins
    fig.add_trace(go.Bar(
        x=champ_ma['MA Period'],
        y=champ_ma['Best on N assets'],
        marker_color='#2962ff',
        text=champ_ma['Best on % assets'],
        textposition='outside', showlegend=False
    ), row=1, col=2)

    # Panel 3: Ichimoku level wins
    fig.add_trace(go.Bar(
        x=champ_ichi['Ichimoku Level'],
        y=champ_ichi['Best on N assets'],
        marker_color='#e91e63',
        text=champ_ichi['Best on % assets'],
        textposition='outside', showlegend=False
    ), row=2, col=1)


    fig.update_layout(
        height=650, template='plotly_dark',
        title=dict(
            text=f'BackTestLAB — Championship Results | {len(multi_df)} assets | Target {TARGET_R}R',
            font=dict(size=16, color='gold')
        )
    )
    fig.show()


plot_multi_asset_championship(multi_df, champ_method, champ_ma, champ_ichi)

---
## 🔗 SECTION 5.1 — Combining All Methods: 1 asset V1






In [34]:
def detect_ma_ichi_confluence(df, ma_col, ichi_level='kijun_sen',
                               tol_pct=0.5, target_r=2.0, prev_bars=5):
    """
    Confluence signal: both MA and ichimoku must simultaneously confirm
    a support or resistance touch at bar i.

    Touch conditions
    ----------------
    MA support    : close > MA  and  low  <= MA*(1+tol)   and context OK
    IIchimoku support : close > ichi  and  low <= ichi*(1+tol)  and context OK
    Both must be True → signal fires.

    Reference level = average(ma_val, Ichi_val)
    Stop            = dynamic MA stop  (MA moves → stop trails price up)
    Target          = entry + target_r * risk   (fixed)
    """
    df   = df.copy()
    tol  = tol_pct / 100.0
    n    = len(df)

    close  = df['close'].values
    high   = df['high'].values
    low    = df['low'].values
    open_  = df['open'].values
    ma     = df[ma_col].values
    ichi  = df[ichi_level].values

    sup_win        = np.zeros(n, dtype=bool)
    sup_loss       = np.zeros(n, dtype=bool)
    sup_incomplete = np.zeros(n, dtype=bool)
    res_win        = np.zeros(n, dtype=bool)
    res_loss       = np.zeros(n, dtype=bool)
    res_incomplete = np.zeros(n, dtype=bool)

    trades    = []
    min_count = prev_bars // 2 + 1

    def context_ok(i, level_arr, direction):
        """
        Check that the majority of prev_bars closes were clearly on
        the correct side of the level (above for support, below for resistance).
        """
        pc = close[i - prev_bars: i]
        pa = level_arr[i - prev_bars: i]
        valid = ~np.isnan(pa)
        if valid.sum() < min_count:
            return False
        if direction == 'sup':
            return np.sum(pc[valid] > pa[valid] * (1 + tol)) >= min_count
        else:
            return np.sum(pc[valid] < pa[valid] * (1 - tol)) >= min_count

    i = max(prev_bars, 1)
    while i < n - 2:
        if np.isnan(ma[i]) or np.isnan(ichi[i]):
            i += 1
            continue

        ma_v  = ma[i]
        ichi_v = ichi[i]

        # ── SUPPORT: both MA and Ichimoku agree ─────────────────────
        ma_sup  = (close[i] > ma_v  and low[i]  <= ma_v  * (1 + tol)
                   and context_ok(i, ma,    'sup'))
        ichi_sup = (close[i] > ichi_v and low[i]  <= ichi_v * (1 + tol)
                   and context_ok(i, ichi, 'sup'))

        if ma_sup and ichi_sup:
            ref_level    = (ma_v + ichi_v) / 2
            entry_idx    = i + 1
            entry_px     = open_[entry_idx]
            initial_stop = ref_level * (1 - tol)
            risk         = entry_px - initial_stop

            if risk > 0:
                target_level = entry_px + target_r * risk

                # Dynamic stop follows MA (the moving component)
                outcome, exit_idx, n_bars = evaluate_sr_outcome(
                    high, low, ma, entry_idx, True, tol, target_level, initial_stop
                )
                exit_stop = ma[exit_idx]*(1-tol) if not np.isnan(ma[exit_idx]) else initial_stop

                trades.append({
                    'type':          'support',
                    'touch_i':       i,
                    'entry_i':       entry_idx,
                    'exit_i':        exit_idx,
                    'n_bars':        n_bars,
                    'ma_val':        round(float(ma_v), 4),
                    'ichi_val':     round(float(ichi_v), 4),
                    'ref_level':     round(float(ref_level), 4),
                    'entry_px':      round(float(entry_px), 4),
                    'initial_stop':  round(float(initial_stop), 4),
                    'exit_stop':     round(float(exit_stop), 4),
                    'target':        round(float(target_level), 4),
                    'risk_pct':      round(float(risk / entry_px * 100), 3),
                    'outcome':       outcome,
                    'exit_px':       round(float(
                                       high[exit_idx] if outcome == 'win'
                                       else low[exit_idx]
                                     ), 4),
                    'touch_date':    df['datetime'].iloc[i],
                    'exit_date':     df['datetime'].iloc[exit_idx],
                })

                if   outcome == 'win':        sup_win[i]        = True
                elif outcome == 'loss':       sup_loss[i]       = True
                else:                         sup_incomplete[i] = True

                i = exit_idx + 1
                continue

        # ── RESISTANCE: both MA and ichimoku agree ──────────────────
        ma_res  = (close[i] < ma_v  and high[i] >= ma_v  * (1 - tol)
                   and context_ok(i, ma,    'res'))
        ichi_res = (close[i] < ichi_v and high[i] >= ichi_v * (1 - tol)
                   and context_ok(i, ichi, 'res'))

        if ma_res and ichi_res:
            ref_level    = (ma_v + ichi_v) / 2
            entry_idx    = i + 1
            entry_px     = open_[entry_idx]
            initial_stop = ref_level * (1 + tol)
            risk         = initial_stop - entry_px

            if risk > 0:
                target_level = entry_px - target_r * risk

                outcome, exit_idx, n_bars = evaluate_sr_outcome(
                    high, low, ma, entry_idx, False, tol, target_level, initial_stop
                )
                exit_stop = ma[exit_idx]*(1+tol) if not np.isnan(ma[exit_idx]) else initial_stop

                trades.append({
                    'type':          'resistance',
                    'touch_i':       i,
                    'entry_i':       entry_idx,
                    'exit_i':        exit_idx,
                    'n_bars':        n_bars,
                    'ma_val':        round(float(ma_v), 4),
                    'ichi_val':     round(float(ichi_v), 4),
                    'ref_level':     round(float(ref_level), 4),
                    'entry_px':      round(float(entry_px), 4),
                    'initial_stop':  round(float(initial_stop), 4),
                    'exit_stop':     round(float(exit_stop), 4),
                    'target':        round(float(target_level), 4),
                    'risk_pct':      round(float(risk / entry_px * 100), 3),
                    'outcome':       outcome,
                    'exit_px':       round(float(
                                       low[exit_idx] if outcome == 'win'
                                       else high[exit_idx]
                                     ), 4),
                    'touch_date':    df['datetime'].iloc[i],
                    'exit_date':     df['datetime'].iloc[exit_idx],
                })

                if   outcome == 'win':        res_win[i]        = True
                elif outcome == 'loss':       res_loss[i]       = True
                else:                         res_incomplete[i] = True

                i = exit_idx + 1
                continue

        i += 1

    df['combo_sup_win']        = sup_win
    df['combo_sup_loss']       = sup_loss
    df['combo_sup_incomplete'] = sup_incomplete
    df['combo_res_win']        = res_win
    df['combo_res_loss']       = res_loss
    df['combo_res_incomplete'] = res_incomplete
    df.attrs['combo_trades']   = trades

    return df


df_main = detect_ma_ichi_confluence(
    df_main,
    ma_col     = best_ma_col,
    ichi_level = best_ichi_level,
    tol_pct    = TOUCH_TOL_PCT,
    target_r   = TARGET_R,
    prev_bars  = PREV_BARS_ABOVE
)

trades_c = df_main.attrs['combo_trades']
dec_c    = [t for t in trades_c if t['outcome'] in ('win','loss')]
wr_c     = round(sum(1 for t in dec_c if t['outcome']=='win') / len(dec_c) * 100, 1) if dec_c else 0
print(f"✅ Confluence ({best_ma_col.upper()} + {best_ichi_level.upper()})")
print(f"   Trades : {len(trades_c)}  |  Decided: {len(dec_c)}  |  Win Rate: {wr_c}%")

✅ Confluence (MA50 + SENKOU_A)
   Trades : 56  |  Decided: 56  |  Win Rate: 39.3%


In [35]:
# ════════════════════════════════════════════════════════════════
# detect_ma_ichi_confluence_v2
# ════════════════════════════════════════════════════════════════

def detect_ma_ichi_confluence_v2(df, ma_list=None, ichi_list=None,
                                   tol_pct=0.5, target_r=2.0, prev_bars=5,
                                   max_proximity_pct=1.0):
    """
    Confluence signal: at least ONE MA from ma_list AND at least ONE Ichimoku
    level from ichi_list must simultaneously confirm a S/R touch at bar i,
    AND the two qualifying levels must be within max_proximity_pct of each other.

    Parameters
    ----------
    ma_list           : list of int periods, e.g. [50, 200]
                        None → all MAs present in df
    ichi_list         : list of column names, e.g. ['kijun_sen', 'senkou_a']
                        None → all Ichimoku levels present in df
    max_proximity_pct : float — max distance (%) between the qualifying MA
                        and the qualifying Ichimoku level.
                        Ensures the two levels form a real confluence ZONE,
                        not just two coincidentally triggered conditions.

    Signal mechanics
    ----------------
    Among all qualifying pairs, the CLOSEST pair is selected (highest quality).
    Reference level = average(ma_val, ichi_val) of the best pair.
    Stop            = dynamic MA stop using the selected MA column.
    Target          = entry + target_r * risk  (fixed).

    Trade dict extras vs v1
    -----------------------
    ma_col_used     : which MA column fired
    ichi_col_used   : which Ichimoku level fired
    proximity_pct   : distance between the two levels in %
    all_qual_ma     : all qualifying MAs at that bar (for analysis)
    all_qual_ichi   : all qualifying Ichimoku levels at that bar
    """
    df  = df.copy()
    tol = tol_pct / 100.0
    n   = len(df)

    # ── Build column lists from parameters ───────────────────────
    _all_ma_periods = [20, 50, 100, 200]
    _all_ichi_lvls  = ['kijun_sen', 'senkou_a', 'senkou_b']

    ma_cols = (
        [f'ma{p}' for p in ma_list if f'ma{p}' in df.columns]
        if ma_list is not None else
        [f'ma{p}' for p in _all_ma_periods if f'ma{p}' in df.columns]
    )
    ichi_cols = (
        [lv for lv in ichi_list if lv in df.columns]
        if ichi_list is not None else
        [lv for lv in _all_ichi_lvls if lv in df.columns]
    )

    if not ma_cols:
        raise ValueError("No valid MA columns found in df. Run add_moving_averages first.")
    if not ichi_cols:
        raise ValueError("No valid Ichimoku columns found in df. Run compute_ichimoku first.")

    close  = df['close'].values
    high   = df['high'].values
    low    = df['low'].values
    open_  = df['open'].values

    ma_arrays   = {col: df[col].values for col in ma_cols}
    ichi_arrays = {col: df[col].values for col in ichi_cols}

    sup_win        = np.zeros(n, dtype=bool)
    sup_loss       = np.zeros(n, dtype=bool)
    sup_incomplete = np.zeros(n, dtype=bool)
    res_win        = np.zeros(n, dtype=bool)
    res_loss       = np.zeros(n, dtype=bool)
    res_incomplete = np.zeros(n, dtype=bool)

    trades    = []
    min_count = prev_bars // 2 + 1

    # ── Touch checker for a single level ─────────────────────────
    def is_touching(i, arr, direction):
        """
        Return (True, level_value) if bar i qualifies as a touch
        for the given direction ('sup' or 'res'), else (False, None).
        """
        lv = arr[i]
        if np.isnan(lv):
            return False, None

        pc    = close[max(0, i - prev_bars): i]
        pa    = arr[max(0, i - prev_bars): i]
        valid = ~np.isnan(pa)
        if valid.sum() < min_count:
            return False, None

        if direction == 'sup':
            ctx = np.sum(pc[valid] > pa[valid] * (1 + tol)) >= min_count
            hit = close[i] > lv and low[i] <= lv * (1 + tol)
        else:
            ctx = np.sum(pc[valid] < pa[valid] * (1 - tol)) >= min_count
            hit = close[i] < lv and high[i] >= lv * (1 - tol)

        return (hit and ctx), lv

    # ── Main loop ─────────────────────────────────────────────────
    i = max(prev_bars, 1)
    while i < n - 2:

        fired = False

        for direction in ('sup', 'res'):
            is_sup = (direction == 'sup')

            # ── Collect all qualifying levels ─────────────────────
            qual_ma   = {}   # col → level_value
            qual_ichi = {}

            for col, arr in ma_arrays.items():
                ok, lv = is_touching(i, arr, direction)
                if ok: qual_ma[col] = lv

            for col, arr in ichi_arrays.items():
                ok, lv = is_touching(i, arr, direction)
                if ok: qual_ichi[col] = lv

            if not qual_ma or not qual_ichi:
                continue

            # ── Find best pair (closest proximity within threshold) ─
            best_pair = None
            best_dist = float('inf')

            for ma_c, ma_lv in qual_ma.items():
                for ic_c, ic_lv in qual_ichi.items():
                    dist = abs(ma_lv - ic_lv) / max(ma_lv, 1e-9) * 100
                    if dist <= max_proximity_pct and dist < best_dist:
                        best_dist = dist
                        best_pair = (ma_c, ma_lv, ic_c, ic_lv)

            if best_pair is None:
                # Levels qualify individually but are too far apart
                continue

            ma_c, ma_lv, ic_c, ic_lv = best_pair
            ref_level = (ma_lv + ic_lv) / 2
            entry_idx = i + 1
            entry_px  = open_[entry_idx]

            if is_sup:
                initial_stop = ref_level * (1 - tol)
                risk         = entry_px - initial_stop
                if risk <= 0: continue
                target_level = entry_px + target_r * risk
            else:
                initial_stop = ref_level * (1 + tol)
                risk         = initial_stop - entry_px
                if risk <= 0: continue
                target_level = entry_px - target_r * risk

            # Dynamic stop follows the selected MA
            ma_arr_used = ma_arrays[ma_c]
            outcome, exit_idx, n_bars = evaluate_sr_outcome(
                high, low, ma_arr_used, entry_idx, is_sup, tol, target_level
            )
            exit_stop = (ma_arr_used[exit_idx] * (1 - tol if is_sup else 1 + tol)
                         if not np.isnan(ma_arr_used[exit_idx]) else initial_stop)

            trades.append({
                'type':           'support' if is_sup else 'resistance',
                'ma_col_used':    ma_c,
                'ichi_col_used':  ic_c,
                'proximity_pct':  round(best_dist, 3),
                'all_qual_ma':    list(qual_ma.keys()),
                'all_qual_ichi':  list(qual_ichi.keys()),
                'touch_i':        i,
                'entry_i':        entry_idx,
                'exit_i':         exit_idx,
                'n_bars':         n_bars,
                'ma_val':         round(float(ma_lv), 4),
                'ichi_val':       round(float(ic_lv), 4),
                'ref_level':      round(float(ref_level), 4),
                'entry_px':       round(float(entry_px), 4),
                'initial_stop':   round(float(initial_stop), 4),
                'exit_stop':      round(float(exit_stop), 4),
                'target':         round(float(target_level), 4),
                'risk_pct':       round(float(risk / entry_px * 100), 3),
                'outcome':        outcome,
                'exit_px':        round(float(
                                     high[exit_idx] if outcome == 'win' and is_sup
                                     else low[exit_idx]  if outcome == 'win'
                                     else low[exit_idx]  if is_sup
                                     else high[exit_idx]
                                   ), 4),
                'touch_date':     df['datetime'].iloc[i],
                'exit_date':      df['datetime'].iloc[exit_idx],
            })

            if   outcome == 'win':
                (sup_win if is_sup else res_win)[i]               = True
            elif outcome == 'loss':
                (sup_loss if is_sup else res_loss)[i]             = True
            else:
                (sup_incomplete if is_sup else res_incomplete)[i] = True

            i     = exit_idx + 1
            fired = True
            break   # one direction per bar

        if not fired:
            i += 1

    df['combo_sup_win']        = sup_win
    df['combo_sup_loss']       = sup_loss
    df['combo_sup_incomplete'] = sup_incomplete
    df['combo_res_win']        = res_win
    df['combo_res_loss']       = res_loss
    df['combo_res_incomplete'] = res_incomplete
    df.attrs['combo_trades']   = trades

    return df


# ── Run ───────────────────────────────────────────────────────────
df_main = detect_ma_ichi_confluence_v2(
    df_main,
    ma_list             = MA_COMBO_LIST,
    ichi_list           = ICHI_COMBO_LIST,
    tol_pct             = TOUCH_TOL_PCT,
    target_r            = TARGET_R,
    prev_bars           = PREV_BARS_ABOVE,
    max_proximity_pct   = COMBO_PROXIMITY_PCT
)

trades_c = df_main.attrs['combo_trades']
dec_c    = [t for t in trades_c if t['outcome'] in ('win','loss')]
wr_c     = round(sum(1 for t in dec_c if t['outcome']=='win') / len(dec_c)*100, 1) if dec_c else 0

# Which pairs fired the most?
from collections import Counter
pair_counter = Counter(f"{t['ma_col_used']}+{t['ichi_col_used']}" for t in trades_c)
print(f"✅ Confluence v2 ({[f'MA{p}' for p in MA_COMBO_LIST]} × {ICHI_COMBO_LIST})")
print(f"   Trades   : {len(trades_c)}  |  Decided: {len(dec_c)}  |  Win Rate: {wr_c}%")
print(f"   Pairs used: {dict(pair_counter.most_common(5))}")

NameError: name 'MA_COMBO_LIST' is not defined

In [115]:
# ── Score ─────────────────────────────────────────────────────────

def score_combo(df, best_ma_col, best_ichi_level, target_r):
    trades  = df.attrs.get('combo_trades', [])
    decided = [t for t in trades if t['outcome'] in ('win','loss')]
    wins    = [t for t in decided if t['outcome'] == 'win']
    losses  = [t for t in decided if t['outcome'] == 'loss']
    incompl = [t for t in trades   if t['outcome'] == 'incomplete']
    wr      = round(len(wins)/len(decided)*100, 1) if decided else 0
    ab      = round(np.mean([t['n_bars'] for t in decided]), 1) if decided else 0

    rows = [
        ('WIN  ✅',           int(len(wins))),
        ('LOSS ❌',           int(len(losses))),
        ('Incomplete ⏳',    int(len(incompl))),
        ('Total Decided',     int(len(decided))),
        ('Win Rate %',        f'{wr:.1f}%'),
        ('Expectancy',        f'{round(wr/100*target_r-(1-wr/100),2):+.2f}R'),
        ('Avg Bars to Exit',  ab),
    ]
    display(pd.DataFrame(rows, columns=['Metric','Value']).style
        .set_caption(f'📊 Combo: {best_ma_col.upper()} + {best_ichi_level.upper()} | Target {target_r}R'))
    return wr

combo_wr = score_combo(df_main, best_ma_col, best_ichi_level, TARGET_R)

,Metric,Value
0,WIN ✅,22
1,LOSS ❌,34
2,Incomplete ⏳,0
3,Total Decided,56
4,Win Rate %,39.3%
5,Expectancy,+0.18R
6,Avg Bars to Exit,4.600000


In [116]:
# ── Plot ─────────────────────────────────────────────────────────

def plot_combo_sr(df, ma_col, ichi_col, symbol, mode='both',
                  last_n_bars=500, show_debug=True):
    df_plot    = df.tail(last_n_bars).copy()
    first_dt   = df_plot['datetime'].iloc[0]
    trades_in  = [t for t in df.attrs.get('combo_trades',[])
                  if t['touch_date'] >= first_dt]
    tlookup    = {t['touch_date']: t for t in trades_in}
    tol        = TOUCH_TOL_PCT / 100.0

    fig = go.Figure()

    fig.add_trace(go.Candlestick(
        x=df_plot['datetime'], open=df_plot['open'], high=df_plot['high'],
        low=df_plot['low'], close=df_plot['close'], name='Price',
        increasing_line_color='#26a69a', decreasing_line_color='#ef5350'
    ))

    # MA + tolerance zone
    fig.add_trace(go.Scatter(
        x=df_plot['datetime'], y=df_plot[ma_col] * (1 + tol),
        line=dict(width=0), showlegend=False, hoverinfo='skip'
    ))
    fig.add_trace(go.Scatter(
        x=df_plot['datetime'], y=df_plot[ma_col] * (1 - tol),
        line=dict(width=0), fill='tonexty',
        fillcolor='rgba(41,98,255,0.10)', name=f'MA zone', hoverinfo='skip'
    ))
    fig.add_trace(go.Scatter(
        x=df_plot['datetime'], y=df_plot[ma_col],
        line=dict(color='#2962ff', width=2), name=ma_col.upper()
    ))

    # Ichimoku
    fig.add_trace(go.Scatter(
        x=df_plot['datetime'], y=df_plot[ichi_col],
        line=dict(color='#e91e63', width=2, dash='dash'), name=ichi_col.upper()
    ))

    # Markers
    marker_cfg = [
        ('combo_sup_win',        'triangle-up',   '#00e676', 'low',  0.997, '✅ Support WIN'),
        ('combo_sup_loss',       'x',             '#ff1744', 'low',  0.997, '❌ Support LOSS'),
        ('combo_sup_incomplete', 'circle',        '#888888', 'low',  0.997, '⏳ Incomplete'),
        ('combo_res_win',        'triangle-down', '#00e676', 'high', 1.003, '✅ Resistance WIN'),
        ('combo_res_loss',       'x',             '#ff1744', 'high', 1.003, '❌ Resistance LOSS'),
        ('combo_res_incomplete', 'circle',        '#888888', 'high', 1.003, '⏳ Incomplete'),
    ]
    for col, sym, color, pcol, off, name in marker_cfg:
        sr_type = 'sup' if 'sup' in col else 'res'
        if sr_type == 'sup' and mode not in ('support','both'):    continue
        if sr_type == 'res' and mode not in ('resistance','both'): continue
        if col not in df_plot.columns: continue
        pts = df_plot[df_plot[col]].copy()
        if pts.empty: continue

        hover = []
        for _, row in pts.iterrows():
            t = tlookup.get(row['datetime'], {})
            if t:
                hover.append(
                    f"<b>COMBO {t['type'].upper()} {t['outcome'].upper()}</b><br>"
                    f"MA level    : {t['ma_val']}<br>"
                    f"Ichimoku level : {t['ichi_val']}<br>"
                    f"Ref level   : {t['ref_level']}<br>"
                    f"Entry       : {t['entry_px']}<br>"
                    f"Init stop   : {t['initial_stop']}<br>"
                    f"Exit stop   : {t['exit_stop']}<br>"
                    f"Target {TARGET_R}R  : {t['target']}<br>"
                    f"Risk        : {t['risk_pct']}%<br>"
                    f"Bars to exit: {t['n_bars']}<br>"
                    f"Exit date   : {t['exit_date'].strftime('%Y-%m-%d')}"
                )
            else:
                hover.append(name)

        fig.add_trace(go.Scatter(
            x=pts['datetime'], y=pts[pcol]*off, mode='markers',
            marker=dict(symbol=sym, size=13, color=color,
                        line=dict(color='white', width=0.8)),
            name=name, text=hover,
            hovertemplate='%{text}<extra></extra>'
        ))

    # Entry / stop / target lines
    oc = {'win':{'t':'#69ff47','e':'#29b6f6','s':'#ff4444'},
          'loss':{'t':'#444','e':'#888','s':'#ff1744'},
          'incomplete':{'t':'#333','e':'#444','s':'#444'}}
    for t in trades_in:
        c=oc[t['outcome']]; x0,x1=t['touch_date'],t['exit_date']
        iw,il=t['outcome']=='win',t['outcome']=='loss'
        fig.add_shape(type='line',x0=x0,x1=x1,y0=t['target'],y1=t['target'],
            line=dict(color=c['t'],width=1.5 if iw else .8,dash='dot'),opacity=.9 if iw else .3)
        fig.add_shape(type='line',x0=x0,x1=x1,y0=t['entry_px'],y1=t['entry_px'],
            line=dict(color=c['e'],width=1.,dash='dash'),opacity=.7 if iw else .3)
        fig.add_shape(type='line',x0=x0,x1=x1,y0=t['initial_stop'],y1=t['initial_stop'],
            line=dict(color=c['s'],width=1.,dash='dash'),opacity=.8 if il else .3)
        if iw:
            fig.add_annotation(x=x1,y=t['target'],text=f"{TARGET_R}R={t['target']:.0f}",
                showarrow=False,font=dict(size=9,color=c['t']),xanchor='left')

    title_text = (f'BackTestLAB — {symbol}  |  {ma_col.upper()} + {ichi_col.upper()} Confluence'
                  f'  |  Target {TARGET_R}R  |  Mode: {mode}')
    fig.update_layout(
        height=700, template='plotly_dark',
        title=dict(text=title_text, font=dict(size=14, color='gold')),
        xaxis_rangeslider_visible=False,
        legend=dict(orientation='h', y=1.05),
        hoverlabel=dict(bgcolor='#1e1e2e', font_size=11)
    )
    fig.update_xaxes(rangebreaks=[dict(bounds=['sat','mon'])])
    fig.show()

    if show_debug and trades_in:
        debug = pd.DataFrame([{
            'Type':         t['type'],
            'Touch Date':   t['touch_date'].strftime('%Y-%m-%d'),
            'MA Level':     t['ma_val'],
            'ichi Level':  t['ichi_val'],
            'Ref Level':    t['ref_level'],
            'Entry':        t['entry_px'],
            'Init Stop':    t['initial_stop'],
            'Exit Stop':    t['exit_stop'],
            f'Target {TARGET_R}R': t['target'],
            'Risk %':       f"{t['risk_pct']}%",
            'Outcome':      t['outcome'].upper(),
            'Exit Date':    t['exit_date'].strftime('%Y-%m-%d'),
            'Bars to Exit': t['n_bars'],
        } for t in trades_in])
        def cr(row):
            if row['Outcome']=='WIN':    return ['background-color:#1a3a1a;color:#b5f5a0']*len(row)
            elif row['Outcome']=='LOSS': return ['background-color:#3a1a1a;color:#f5a0a0']*len(row)
            else:                        return ['background-color:#2a2a2a;color:#aaaaaa']*len(row)
        display(debug.style.apply(cr,axis=1)
            .set_caption(f'📋 Combo Trades — Last {last_n_bars} bars'))


plot_combo_sr(df_main, best_ma_col, best_ichi_level, DEMO_LABEL, mode=SR_DISPLAY_MODE, last_n_bars=500)


,Type,Touch Date,MA Level,ichi Level,Ref Level,Entry,Init Stop,Exit Stop,Target 2.0R,Risk %,Outcome,Exit Date,Bars to Exit
0,support,2024-05-30,2318.856000,2324.600000,2321.728000,2344.100100,2310.119300,2326.867200,2412.061600,1.45%,LOSS,2024-06-07,5
1,resistance,2024-11-29,2666.964000,2669.850000,2668.407000,2649.000000,2681.749000,2681.384200,2583.502000,1.236%,LOSS,2024-12-10,6
2,resistance,2025-01-03,2660.874000,2651.975000,2656.424500,2645.500000,2669.706600,2669.543300,2597.086800,0.915%,LOSS,2025-01-08,2
3,support,2025-07-17,3321.932000,3299.575000,3310.753500,3338.200000,3294.199700,3308.918300,3426.200400,1.318%,WIN,2025-07-22,2


## 🔗 SECTION 5.2 — Combining All Methods: 1 asset V2

In [36]:
# ════════════════════════════════════════════════════════════════
# SECTION 5.2 — Config: choose your confluence combination
# ════════════════════════════════════════════════════════════════

# ── Option A: all available levels ───────────────────────────────
MA_COMBO_LIST   = MA_PERIODS                                  # [20, 50, 100, 200]
ICHI_COMBO_LIST = ['kijun_sen', 'senkou_a', 'senkou_b']

# ── Option B: specific levels only ───────────────────────────────
# MA_COMBO_LIST   = [50, 200]
# ICHI_COMBO_LIST = ['kijun_sen']

# ── Option C: use best found in previous sections ─────────────────
# MA_COMBO_LIST   = [int(best_ma_col.replace('ma', ''))]
# ICHI_COMBO_LIST = [best_ichi_level]

# ── Proximity filter: max distance between MA and Ichimoku levels ─
# If MA100=5100 and Kijun=5115 → distance = 0.29% → strong confluence
# If MA20=5100  and Senkou_B=5250 → distance = 2.9% → too far apart
COMBO_PROXIMITY_PCT = 1.0   # levels must be within 1% of each other

print(f"Confluence config:")
print(f"  MA levels         : {[f'MA{p}' for p in MA_COMBO_LIST]}")
print(f"  Ichimoku levels   : {ICHI_COMBO_LIST}")
print(f"  Max proximity     : {COMBO_PROXIMITY_PCT}%")
print(f"  → Tests {len(MA_COMBO_LIST) * len(ICHI_COMBO_LIST)} MA×Ichimoku pairs per bar")

Confluence config:
  MA levels         : ['MA20', 'MA50', 'MA100', 'MA200']
  Ichimoku levels   : ['kijun_sen', 'senkou_a', 'senkou_b']
  Max proximity     : 1.0%
  → Tests 12 MA×Ichimoku pairs per bar


In [38]:
# ════════════════════════════════════════════════════════════════
# detect_ma_ichi_confluence_v2
# ════════════════════════════════════════════════════════════════

def detect_ma_ichi_confluence_v2(df, ma_list=None, ichi_list=None,
                                   tol_pct=0.5, target_r=2.0, prev_bars=5,
                                   max_proximity_pct=1.0):
    """
    Confluence signal: at least ONE MA from ma_list AND at least ONE Ichimoku
    level from ichi_list must simultaneously confirm a S/R touch at bar i,
    AND the two qualifying levels must be within max_proximity_pct of each other.

    Parameters
    ----------
    ma_list           : list of int periods, e.g. [50, 200]
                        None → all MAs present in df
    ichi_list         : list of column names, e.g. ['kijun_sen', 'senkou_a']
                        None → all Ichimoku levels present in df
    max_proximity_pct : float — max distance (%) between the qualifying MA
                        and the qualifying Ichimoku level.
                        Ensures the two levels form a real confluence ZONE,
                        not just two coincidentally triggered conditions.

    Signal mechanics
    ----------------
    Among all qualifying pairs, the CLOSEST pair is selected (highest quality).
    Reference level = average(ma_val, ichi_val) of the best pair.
    Stop            = dynamic MA stop using the selected MA column.
    Target          = entry + target_r * risk  (fixed).

    Trade dict extras vs v1
    -----------------------
    ma_col_used     : which MA column fired
    ichi_col_used   : which Ichimoku level fired
    proximity_pct   : distance between the two levels in %
    all_qual_ma     : all qualifying MAs at that bar (for analysis)
    all_qual_ichi   : all qualifying Ichimoku levels at that bar
    """
    df  = df.copy()
    tol = tol_pct / 100.0
    n   = len(df)

    # ── Build column lists from parameters ───────────────────────
    _all_ma_periods = [20, 50, 100, 200]
    _all_ichi_lvls  = ['kijun_sen', 'senkou_a', 'senkou_b']

    ma_cols = (
        [f'ma{p}' for p in ma_list if f'ma{p}' in df.columns]
        if ma_list is not None else
        [f'ma{p}' for p in _all_ma_periods if f'ma{p}' in df.columns]
    )
    ichi_cols = (
        [lv for lv in ichi_list if lv in df.columns]
        if ichi_list is not None else
        [lv for lv in _all_ichi_lvls if lv in df.columns]
    )

    if not ma_cols:
        raise ValueError("No valid MA columns found in df. Run add_moving_averages first.")
    if not ichi_cols:
        raise ValueError("No valid Ichimoku columns found in df. Run compute_ichimoku first.")

    close  = df['close'].values
    high   = df['high'].values
    low    = df['low'].values
    open_  = df['open'].values

    ma_arrays   = {col: df[col].values for col in ma_cols}
    ichi_arrays = {col: df[col].values for col in ichi_cols}

    sup_win        = np.zeros(n, dtype=bool)
    sup_loss       = np.zeros(n, dtype=bool)
    sup_incomplete = np.zeros(n, dtype=bool)
    res_win        = np.zeros(n, dtype=bool)
    res_loss       = np.zeros(n, dtype=bool)
    res_incomplete = np.zeros(n, dtype=bool)

    trades    = []
    min_count = prev_bars // 2 + 1

    # ── Touch checker for a single level ─────────────────────────
    def is_touching(i, arr, direction):
        """
        Return (True, level_value) if bar i qualifies as a touch
        for the given direction ('sup' or 'res'), else (False, None).
        """
        lv = arr[i]
        if np.isnan(lv):
            return False, None

        pc    = close[max(0, i - prev_bars): i]
        pa    = arr[max(0, i - prev_bars): i]
        valid = ~np.isnan(pa)
        if valid.sum() < min_count:
            return False, None

        if direction == 'sup':
            ctx = np.sum(pc[valid] > pa[valid] * (1 + tol)) >= min_count
            hit = close[i] > lv and low[i] <= lv * (1 + tol)
        else:
            ctx = np.sum(pc[valid] < pa[valid] * (1 - tol)) >= min_count
            hit = close[i] < lv and high[i] >= lv * (1 - tol)

        return (hit and ctx), lv

    # ── Main loop ─────────────────────────────────────────────────
    i = max(prev_bars, 1)
    while i < n - 2:

        fired = False

        for direction in ('sup', 'res'):
            is_sup = (direction == 'sup')

            # ── Collect all qualifying levels ─────────────────────
            qual_ma   = {}   # col → level_value
            qual_ichi = {}

            for col, arr in ma_arrays.items():
                ok, lv = is_touching(i, arr, direction)
                if ok: qual_ma[col] = lv

            for col, arr in ichi_arrays.items():
                ok, lv = is_touching(i, arr, direction)
                if ok: qual_ichi[col] = lv

            if not qual_ma or not qual_ichi:
                continue

            # ── Find best pair (closest proximity within threshold) ─
            best_pair = None
            best_dist = float('inf')

            for ma_c, ma_lv in qual_ma.items():
                for ic_c, ic_lv in qual_ichi.items():
                    dist = abs(ma_lv - ic_lv) / max(ma_lv, 1e-9) * 100
                    if dist <= max_proximity_pct and dist < best_dist:
                        best_dist = dist
                        best_pair = (ma_c, ma_lv, ic_c, ic_lv)

            if best_pair is None:
                # Levels qualify individually but are too far apart
                continue

            ma_c, ma_lv, ic_c, ic_lv = best_pair
            ref_level = (ma_lv + ic_lv) / 2
            entry_idx = i + 1
            entry_px  = open_[entry_idx]

            if is_sup:
                initial_stop = ref_level * (1 - tol)
                risk         = entry_px - initial_stop
                if risk <= 0: continue
                target_level = entry_px + target_r * risk
            else:
                initial_stop = ref_level * (1 + tol)
                risk         = initial_stop - entry_px
                if risk <= 0: continue
                target_level = entry_px - target_r * risk

            # Dynamic stop follows the selected MA
            ma_arr_used = ma_arrays[ma_c]
            outcome, exit_idx, n_bars = evaluate_sr_outcome(
                high, low, ma_arr_used, entry_idx, is_sup, tol, target_level, initial_stop
            )
            exit_stop = (ma_arr_used[exit_idx] * (1 - tol if is_sup else 1 + tol)
                         if not np.isnan(ma_arr_used[exit_idx]) else initial_stop)

            trades.append({
                'type':           'support' if is_sup else 'resistance',
                'ma_col_used':    ma_c,
                'ichi_col_used':  ic_c,
                'proximity_pct':  round(best_dist, 3),
                'all_qual_ma':    list(qual_ma.keys()),
                'all_qual_ichi':  list(qual_ichi.keys()),
                'touch_i':        i,
                'entry_i':        entry_idx,
                'exit_i':         exit_idx,
                'n_bars':         n_bars,
                'ma_val':         round(float(ma_lv), 4),
                'ichi_val':       round(float(ic_lv), 4),
                'ref_level':      round(float(ref_level), 4),
                'entry_px':       round(float(entry_px), 4),
                'initial_stop':   round(float(initial_stop), 4),
                'exit_stop':      round(float(exit_stop), 4),
                'target':         round(float(target_level), 4),
                'risk_pct':       round(float(risk / entry_px * 100), 3),
                'outcome':        outcome,
                'exit_px':        round(float(
                                     high[exit_idx] if outcome == 'win' and is_sup
                                     else low[exit_idx]  if outcome == 'win'
                                     else low[exit_idx]  if is_sup
                                     else high[exit_idx]
                                   ), 4),
                'touch_date':     df['datetime'].iloc[i],
                'exit_date':      df['datetime'].iloc[exit_idx],
            })

            if   outcome == 'win':
                (sup_win if is_sup else res_win)[i]               = True
            elif outcome == 'loss':
                (sup_loss if is_sup else res_loss)[i]             = True
            else:
                (sup_incomplete if is_sup else res_incomplete)[i] = True

            i     = exit_idx + 1
            fired = True
            break   # one direction per bar

        if not fired:
            i += 1

    df['combo_sup_win']        = sup_win
    df['combo_sup_loss']       = sup_loss
    df['combo_sup_incomplete'] = sup_incomplete
    df['combo_res_win']        = res_win
    df['combo_res_loss']       = res_loss
    df['combo_res_incomplete'] = res_incomplete
    df.attrs['combo_trades']   = trades

    return df


# ── Run ───────────────────────────────────────────────────────────
df_main = detect_ma_ichi_confluence_v2(
    df_main,
    ma_list             = MA_COMBO_LIST,
    ichi_list           = ICHI_COMBO_LIST,
    tol_pct             = TOUCH_TOL_PCT,
    target_r            = TARGET_R,
    prev_bars           = PREV_BARS_ABOVE,
    max_proximity_pct   = COMBO_PROXIMITY_PCT
)

trades_c = df_main.attrs['combo_trades']
dec_c    = [t for t in trades_c if t['outcome'] in ('win','loss')]
wr_c     = round(sum(1 for t in dec_c if t['outcome']=='win') / len(dec_c)*100, 1) if dec_c else 0

# Which pairs fired the most?
from collections import Counter
pair_counter = Counter(f"{t['ma_col_used']}+{t['ichi_col_used']}" for t in trades_c)
print(f"✅ Confluence v2 ({[f'MA{p}' for p in MA_COMBO_LIST]} × {ICHI_COMBO_LIST})")
print(f"   Trades   : {len(trades_c)}  |  Decided: {len(dec_c)}  |  Win Rate: {wr_c}%")
print(f"   Pairs used: {dict(pair_counter.most_common(5))}")

✅ Confluence v2 (['MA20', 'MA50', 'MA100', 'MA200'] × ['kijun_sen', 'senkou_a', 'senkou_b'])
   Trades   : 198  |  Decided: 198  |  Win Rate: 32.3%
   Pairs used: {'ma20+kijun_sen': 86, 'ma50+senkou_a': 28, 'ma100+senkou_b': 21, 'ma50+kijun_sen': 16, 'ma20+senkou_a': 9}


In [41]:
def score_combo(df, ma_list, ichi_list, target_r):
    trades  = df.attrs.get('combo_trades', [])
    decided = [t for t in trades if t['outcome'] in ('win','loss')]
    wins    = [t for t in decided if t['outcome'] == 'win']
    losses  = [t for t in decided if t['outcome'] == 'loss']
    incompl = [t for t in trades   if t['outcome'] == 'incomplete']
    wr      = round(len(wins)/len(decided)*100, 1) if decided else 0
    ab      = round(np.mean([t['n_bars'] for t in decided]), 1) if decided else 0
    avg_prox= round(np.mean([t['proximity_pct'] for t in trades]), 3) if trades else 0

    # ── Global summary ────────────────────────────────────────────
    ma_lbl   = f"MA[{','.join(str(p) for p in sorted(ma_list))}]"
    ichi_lbl = f"Ichi[{','.join(lv.replace('kijun_sen','Kijun').replace('senkou_a','SenkouA').replace('senkou_b','SenkouB') for lv in ichi_list)}]"

    rows = [
        ('WIN  ✅',              int(len(wins))),
        ('LOSS ❌',              int(len(losses))),
        ('Incomplete ⏳',       int(len(incompl))),
        ('Total Decided',        int(len(decided))),
        ('Win Rate %',           f'{wr:.1f}%'),
        ('Expectancy',           f'{round(wr/100*target_r-(1-wr/100),2):+.2f}R'),
        ('Avg Bars to Exit',     ab),
        ('Avg Proximity %',      f'{avg_prox:.3f}%'),
    ]
    display(pd.DataFrame(rows, columns=['Metric', 'Value']).style
        .set_caption(f'📊 Combo: {ma_lbl} × {ichi_lbl} | Proximity ≤{COMBO_PROXIMITY_PCT}% | Target {target_r}R'))

    # ── Breakdown by pair ─────────────────────────────────────────
    pair_rows = []
    pairs = sorted(set(
        (t['ma_col_used'], t['ichi_col_used']) for t in trades
    ))
    for ma_c, ic_c in pairs:
        sub     = [t for t in trades   if t['ma_col_used']==ma_c and t['ichi_col_used']==ic_c]
        sub_dec = [t for t in sub      if t['outcome'] in ('win','loss')]
        sub_w   = sum(1 for t in sub_dec if t['outcome']=='win')
        sub_wr  = round(sub_w/len(sub_dec)*100,1) if sub_dec else 0
        pair_rows.append({
            'Pair':          f"{ma_c.upper()} + {ic_c.replace('_',' ').title()}",
            'Trades':        len(sub),
            'Decided':       len(sub_dec),
            'Win Rate %':    sub_wr,
            'Expectancy':    f'{round(sub_wr/100*target_r-(1-sub_wr/100),2):+.2f}R',
            'Avg Proximity': f'{round(np.mean([t["proximity_pct"] for t in sub]),3):.3f}%',
        })
    if pair_rows:
        display(pd.DataFrame(pair_rows)
            .sort_values('Win Rate %', ascending=False)
            .style
            .background_gradient(cmap='RdYlGn', subset=['Win Rate %'])
            .hide(axis='index')
            .set_caption('📊 Breakdown by MA × Ichimoku Pair'))

    return wr


# ── Appel ─────────────────────────────────────────────────────────
combo_wr = score_combo(df_main, MA_COMBO_LIST, ICHI_COMBO_LIST, TARGET_R)

,Metric,Value
0,WIN ✅,64
1,LOSS ❌,134
2,Incomplete ⏳,0
3,Total Decided,198
4,Win Rate %,32.3%
5,Expectancy,-0.03R
6,Avg Bars to Exit,4.500000
7,Avg Proximity %,0.291%


Pair,Trades,Decided,Win Rate %,Expectancy,Avg Proximity
MA50 + Senkou B,7,7,71.400000,+1.14R,0.199%
MA200 + Kijun Sen,3,3,66.700000,+1.00R,0.090%
MA100 + Senkou A,6,6,50.000000,+0.50R,0.136%
MA200 + Senkou A,4,4,50.000000,+0.50R,0.216%
MA20 + Senkou A,9,9,44.400000,+0.33R,0.185%
MA20 + Senkou B,8,8,37.500000,+0.12R,0.256%
MA200 + Senkou B,6,6,33.300000,-0.00R,0.183%
MA50 + Senkou A,28,28,32.100000,-0.04R,0.361%
MA100 + Senkou B,21,21,28.600000,-0.14R,0.324%
MA20 + Kijun Sen,86,86,26.700000,-0.20R,0.313%


In [39]:
# ── Updated plot ──────────────────────────────────────────────────

def plot_combo_sr(df, ma_list, ichi_list, symbol, mode='both',
                  last_n_bars=500, show_debug=True):
    """
    Shows ALL MA lines from ma_list and ALL Ichimoku levels from ichi_list.
    Hover tooltip shows which specific pair triggered each signal.
    """
    df_plot   = df.tail(last_n_bars).copy()
    first_dt  = df_plot['datetime'].iloc[0]
    trades_in = [t for t in df.attrs.get('combo_trades', [])
                 if t['touch_date'] >= first_dt]
    tlookup   = {t['touch_date']: t for t in trades_in}
    tol       = TOUCH_TOL_PCT / 100.0

    fig = go.Figure()

    # ── Candlesticks ─────────────────────────────────────────────
    fig.add_trace(go.Candlestick(
        x=df_plot['datetime'], open=df_plot['open'], high=df_plot['high'],
        low=df_plot['low'], close=df_plot['close'], name='Price',
        increasing_line_color='#26a69a', decreasing_line_color='#ef5350'
    ))

    # ── MA lines (blue spectrum, thicker = longer period) ────────
    ma_colors = {
        'ma20':  ('rgba(100,149,237,0.55)', 1.0),
        'ma50':  ('rgba(65,105,225,0.70)',  1.5),
        'ma100': ('#2962ff',                2.0),
        'ma200': ('#0033aa',                2.5),
    }
    for p in sorted(ma_list):
        col = f'ma{p}'
        if col not in df_plot.columns: continue
        color, width = ma_colors.get(col, ('#2962ff', 1.5))
        # Shaded zone around the best/last MA in list
        if p == max(ma_list):
            fig.add_trace(go.Scatter(
                x=df_plot['datetime'], y=df_plot[col] * (1 + tol),
                line=dict(width=0), showlegend=False, hoverinfo='skip'
            ))
            fig.add_trace(go.Scatter(
                x=df_plot['datetime'], y=df_plot[col] * (1 - tol),
                line=dict(width=0), fill='tonexty',
                fillcolor='rgba(41,98,255,0.08)', name='MA zone',
                hoverinfo='skip'
            ))
        fig.add_trace(go.Scatter(
            x=df_plot['datetime'], y=df_plot[col],
            line=dict(color=color, width=width),
            name=f'MA{p}'
        ))

    # ── Ichimoku lines (red/pink spectrum) ───────────────────────
    ichi_styles = {
        'kijun_sen': ('#e91e63', 'solid',  2.0),
        'senkou_a':  ('#f48fb1', 'dash',   1.2),
        'senkou_b':  ('#f8bbd0', 'dot',    1.2),
    }
    for lv in ichi_list:
        if lv not in df_plot.columns: continue
        color, dash, width = ichi_styles.get(lv, ('#e91e63', 'solid', 1.5))
        fig.add_trace(go.Scatter(
            x=df_plot['datetime'], y=df_plot[lv],
            line=dict(color=color, width=width, dash=dash),
            name=lv.replace('_', ' ').title()
        ))

    # ── WIN / LOSS / INCOMPLETE markers ──────────────────────────
    marker_cfg = [
        ('combo_sup_win',        'triangle-up',   '#00e676', 'low',  0.997, '✅ Support WIN'),
        ('combo_sup_loss',       'x',             '#ff1744', 'low',  0.997, '❌ Support LOSS'),
        ('combo_sup_incomplete', 'circle',        '#888888', 'low',  0.997, '⏳ Incomplete'),
        ('combo_res_win',        'triangle-down', '#00e676', 'high', 1.003, '✅ Resistance WIN'),
        ('combo_res_loss',       'x',             '#ff1744', 'high', 1.003, '❌ Resistance LOSS'),
        ('combo_res_incomplete', 'circle',        '#888888', 'high', 1.003, '⏳ Incomplete'),
    ]
    for col, sym, color, pcol, off, name in marker_cfg:
        sr_type = 'sup' if 'sup' in col else 'res'
        if sr_type == 'sup' and mode not in ('support', 'both'):    continue
        if sr_type == 'res' and mode not in ('resistance', 'both'): continue
        if col not in df_plot.columns: continue
        pts = df_plot[df_plot[col]].copy()
        if pts.empty: continue

        hover = []
        for _, row in pts.iterrows():
            t = tlookup.get(row['datetime'], {})
            if t:
                hover.append(
                    f"<b>COMBO {t['type'].upper()} {t['outcome'].upper()}</b><br>"
                    f"Pair used      : {t['ma_col_used'].upper()} + "
                    f"{t['ichi_col_used'].replace('_',' ').title()}<br>"
                    f"Proximity      : {t['proximity_pct']}%<br>"
                    f"MA value       : {t['ma_val']}<br>"
                    f"Ichimoku value : {t['ichi_val']}<br>"
                    f"Ref level      : {t['ref_level']}<br>"
                    f"All qual. MA   : {t['all_qual_ma']}<br>"
                    f"All qual. Ichi : {t['all_qual_ichi']}<br>"
                    f"Entry          : {t['entry_px']}<br>"
                    f"Init stop      : {t['initial_stop']}<br>"
                    f"Exit stop      : {t['exit_stop']}<br>"
                    f"Target {TARGET_R}R     : {t['target']}<br>"
                    f"Risk           : {t['risk_pct']}%<br>"
                    f"Bars to exit   : {t['n_bars']}<br>"
                    f"Exit date      : {t['exit_date'].strftime('%Y-%m-%d')}"
                )
            else:
                hover.append(name)

        fig.add_trace(go.Scatter(
            x=pts['datetime'], y=pts[pcol] * off, mode='markers',
            marker=dict(symbol=sym, size=13, color=color,
                        line=dict(color='white', width=0.8)),
            name=name, text=hover,
            hovertemplate='%{text}<extra></extra>'
        ))

    # ── Entry / Stop / Target lines ──────────────────────────────
    oc = {'win':       {'t': '#69ff47', 'e': '#29b6f6', 's': '#ff4444'},
          'loss':      {'t': '#444',    'e': '#888',    's': '#ff1744'},
          'incomplete':{'t': '#333',    'e': '#444',    's': '#444'}}
    for t in trades_in:
        c  = oc[t['outcome']]
        x0, x1 = t['touch_date'], t['exit_date']
        iw, il  = t['outcome'] == 'win', t['outcome'] == 'loss'
        fig.add_shape(type='line', x0=x0, x1=x1,
            y0=t['target'], y1=t['target'],
            line=dict(color=c['t'], width=1.5 if iw else .8, dash='dot'),
            opacity=.9 if iw else .3)
        fig.add_shape(type='line', x0=x0, x1=x1,
            y0=t['entry_px'], y1=t['entry_px'],
            line=dict(color=c['e'], width=1., dash='dash'),
            opacity=.7 if iw else .3)
        fig.add_shape(type='line', x0=x0, x1=x1,
            y0=t['initial_stop'], y1=t['initial_stop'],
            line=dict(color=c['s'], width=1., dash='dash'),
            opacity=.8 if il else .3)
        if iw:
            fig.add_annotation(
                x=x1, y=t['target'],
                text=f"{TARGET_R}R={t['target']:.0f}",
                showarrow=False, font=dict(size=9, color=c['t']),
                xanchor='left'
            )

    ma_lbl   = f"MA[{','.join(str(p) for p in sorted(ma_list))}]"
    ichi_lbl = f"Ichi[{','.join(lv.replace('_sen','').replace('senkou_','S') for lv in ichi_list)}]"
    title_text = (f'BackTestLAB — {symbol}  |  Confluence {ma_lbl} × {ichi_lbl}'
                  f'  |  Proximity ≤{COMBO_PROXIMITY_PCT}%  |  Target {TARGET_R}R  |  Mode: {mode}')

    fig.update_layout(
        height=700, template='plotly_dark',
        title=dict(text=title_text, font=dict(size=13, color='gold')),
        xaxis_rangeslider_visible=False,
        legend=dict(orientation='h', y=1.06),
        hoverlabel=dict(bgcolor='#1e1e2e', font_size=11)
    )
    fig.update_xaxes(rangebreaks=[dict(bounds=['sat', 'mon'])])
    fig.show()

    # ── Debug table ──────────────────────────────────────────────
    if show_debug and trades_in:
        debug = pd.DataFrame([{
            'Type':          t['type'],
            'Pair':          f"{t['ma_col_used'].upper()} + {t['ichi_col_used'].replace('_',' ').title()}",
            'Proximity %':   f"{t['proximity_pct']}%",
            'Touch Date':    t['touch_date'].strftime('%Y-%m-%d'),
            'MA Val':        t['ma_val'],
            'Ichi Val':      t['ichi_val'],
            'Ref Level':     t['ref_level'],
            'Entry':         t['entry_px'],
            'Init Stop':     t['initial_stop'],
            'Exit Stop':     t['exit_stop'],
            f'Target {TARGET_R}R': t['target'],
            'Risk %':        f"{t['risk_pct']}%",
            'Outcome':       t['outcome'].upper(),
            'Exit Date':     t['exit_date'].strftime('%Y-%m-%d'),
            'Bars to Exit':  t['n_bars'],
        } for t in trades_in])

        def cr(row):
            if row['Outcome'] == 'WIN':    return ['background-color:#1a3a1a;color:#b5f5a0'] * len(row)
            elif row['Outcome'] == 'LOSS': return ['background-color:#3a1a1a;color:#f5a0a0'] * len(row)
            else:                          return ['background-color:#2a2a2a;color:#aaaaaa'] * len(row)

        display(debug.style.apply(cr, axis=1)
            .set_caption(f'📋 Confluence v2 Trades — Last {last_n_bars} bars'))


plot_combo_sr(df_main, MA_COMBO_LIST, ICHI_COMBO_LIST,
              DEMO_LABEL, mode=SR_DISPLAY_MODE, last_n_bars=500)

,Type,Pair,Proximity %,Touch Date,MA Val,Ichi Val,Ref Level,Entry,Init Stop,Exit Stop,Target 2.0R,Risk %,Outcome,Exit Date,Bars to Exit
0,support,MA20 + Senkou A,0.694%,2024-05-28,2348.860000,2332.550000,2340.705000,2340.300000,2329.001500,2341.687700,2362.897200,0.483%,LOSS,2024-05-30,1
1,resistance,MA50 + Kijun Sen,0.117%,2024-07-01,2337.984000,2335.250000,2336.617000,2330.700000,2348.300100,2348.713100,2295.499700,0.755%,LOSS,2024-07-03,1
2,support,MA20 + Kijun Sen,0.239%,2024-07-23,2378.370000,2384.050000,2381.210000,2421.000000,2369.304000,2373.910800,2524.392000,2.135%,LOSS,2024-07-25,1
3,support,MA50 + Senkou B,0.258%,2024-08-05,2362.802000,2356.700100,2359.751000,2414.500000,2347.952300,2434.723200,2547.595400,2.756%,WIN,2024-09-12,26
4,resistance,MA20 + Kijun Sen,0.108%,2024-11-25,2668.715000,2671.600000,2670.157500,2625.600100,2683.508300,2647.054400,2509.783700,2.206%,LOSS,2024-12-10,9
5,support,MA20 + Kijun Sen,0.238%,2024-12-13,2650.300000,2644.000000,2647.150000,2658.300000,2633.914300,2642.993600,2707.071600,0.917%,LOSS,2024-12-17,1
6,resistance,MA50 + Kijun Sen,0.11%,2025-01-03,2660.874000,2657.950100,2659.412000,2645.500000,2672.709100,2669.543300,2591.081800,1.029%,LOSS,2025-01-08,2
7,support,MA20 + Kijun Sen,0.259%,2025-05-02,3226.015000,3217.650000,3221.832500,3242.700000,3205.723300,3247.665100,3316.653200,1.14%,WIN,2025-05-06,1
8,support,MA20 + Senkou A,0.099%,2025-06-06,3291.030000,3294.275000,3292.652500,3315.600100,3276.189300,3293.385300,3394.421800,1.189%,WIN,2025-06-12,3
9,support,MA50 + Kijun Sen,0.105%,2025-06-24,3312.412000,3308.950000,3310.681000,3321.600100,3294.127600,3301.593100,3376.545200,0.827%,LOSS,2025-06-27,2


## 🔗 SECTION 5.3 — Combining All Methods: multi-assets

In [117]:
# ════════════════════════════════════════════════════════════════
# SECTION 5 — Multi-Asset: Does the Combo always win?
# ════════════════════════════════════════════════════════════════

def run_combo_analysis(symbol, label, start, end,
                        ma_periods, tol_pct, target_r, prev_bars):
    """
    Run MA + Ichimoku individual + combo analysis on one asset.
    Returns a summary dict with win rates and expectancies.
    """
    df = getDataFromYahoo(symbol, start, end)
    if df is None or len(df) < 300:
        return None

    # ── Individual methods ────────────────────────────────────────
    df = add_moving_averages(df, ma_periods)
    for p in ma_periods:
        df = detect_ma_sr_touches(df, f'ma{p}', tol_pct, target_r, prev_bars)

    df = compute_ichimoku(df)
    df = detect_ichimoku_sr(df, tol_pct, target_r, prev_bars)

    def wr_exp(w, l, tr):
        rate = round(w/(w+l)*100, 1) if (w+l) > 0 else 0
        return rate, round(rate/100*tr - (1-rate/100), 2)

    # Best MA
    best_ma_wr, best_ma_lbl, best_ma_exp = 0, '', 0
    for p in ma_periods:
        mc = f'ma{p}'
        w = df[f'sr_sup_win_{mc}'].sum() + df[f'sr_res_win_{mc}'].sum()
        l = df[f'sr_sup_loss_{mc}'].sum() + df[f'sr_res_loss_{mc}'].sum()
        rate, exp = wr_exp(w, l, target_r)
        if rate > best_ma_wr:
            best_ma_wr, best_ma_lbl, best_ma_exp = rate, f'MA{p}', exp

    # ichimoku
    lv = best_ichi_level
    w_ki = df[f'ichi_sup_win_{lv}'].sum() + df[f'ichi_res_win_{lv}'].sum()
    l_ki = df[f'ichi_sup_loss_{lv}'].sum() + df[f'ichi_res_loss_{lv}'].sum()
    ichi_wr, ichi_exp = wr_exp(w_ki, l_ki, target_r)

    # ── Combination ───────────────────────────────────────────────
    best_ma_col = f'ma{ma_periods[
        np.argmax([
            float(
                wr_exp(
                    df[f"sr_sup_win_ma{p}"].sum() + df[f"sr_res_win_ma{p}"].sum(),
                    df[f"sr_sup_loss_ma{p}"].sum() + df[f"sr_res_loss_ma{p}"].sum(),
                    target_r
                )[0]
            )
            for p in ma_periods
        ])
    ]}'

    df = detect_ma_ichi_confluence(
        df, best_ma_col, best_ichi_level,
        tol_pct=tol_pct, target_r=target_r, prev_bars=prev_bars
    )

    combo_trades = df.attrs.get('combo_trades', [])
    combo_dec    = [t for t in combo_trades if t['outcome'] in ('win','loss')]
    cw           = sum(1 for t in combo_dec if t['outcome'] == 'win')
    cl           = len(combo_dec) - cw
    combo_wr, combo_exp = wr_exp(cw, cl, target_r)

    # ── Winner per asset ──────────────────────────────────────────
    rates   = {'MA': best_ma_wr, 'ichi': ichi_wr, 'Combo': combo_wr}
    winner  = max(rates, key=rates.get)
    combo_beats_both = combo_wr > best_ma_wr and combo_wr > ichi_wr

    return {
        'Asset':            label,
        # Individual
        'Best MA':          best_ma_lbl,
        'MA Win Rate %':    best_ma_wr,
        'MA Expectancy':    best_ma_exp,
        'ichi Win Rate %': ichi_wr,
        'ichi Expectancy': ichi_exp,
        # Combo
        'Combo Win Rate %': combo_wr,
        'Combo Expectancy': combo_exp,
        'Combo Trades':     len(combo_trades),
        'Combo Decided':    len(combo_dec),
        # Verdict
        'Winner':           winner,
        'Combo Wins?':      '✅ Yes' if combo_beats_both else '❌ No',
    }


print(f"⏳ Multi-asset combo analysis | Target {TARGET_R}R ...")
combo_multi = []
for label, symbol in ASSETS.items():
    print(f"  {label} ({symbol})...", end=" ")
    r = run_combo_analysis(
        symbol, label, START_DATE, END_DATE,
        MA_PERIODS, TOUCH_TOL_PCT, TARGET_R, PREV_BARS_ABOVE
    )
    if r:
        combo_multi.append(r)
        print(f"✅  MA: {r['MA Win Rate %']}%  "
              f"ichi: {r['ichi Win Rate %']}%  "
              f"Combo: {r['Combo Win Rate %']}%  "
              f"→ Winner: {r['Winner']}")
    else:
        print("⚠️  skipped")

combo_multi_df = pd.DataFrame(combo_multi)
print(f"\n✅ Done — {len(combo_multi_df)} assets")

⏳ Multi-asset combo analysis | Target 2.0R ...
  Apple (AAPL)... ✅  MA: 36.5%  ichi: 35.0%  Combo: 25.0%  → Winner: MA
  Microsoft (MSFT)... ✅  MA: 42.4%  ichi: 35.5%  Combo: 31.6%  → Winner: MA
  Nvidia (NVDA)... ✅  MA: 38.6%  ichi: 27.5%  Combo: 32.1%  → Winner: MA
  Alphabet (GOOGL)... ✅  MA: 37.9%  ichi: 34.5%  Combo: 52.4%  → Winner: Combo
  Meta (META)... ✅  MA: 37.2%  ichi: 31.6%  Combo: 16.1%  → Winner: MA
  Amazon (AMZN)... ✅  MA: 36.3%  ichi: 32.5%  Combo: 45.5%  → Winner: Combo
  Oracle (ORCL)... ✅  MA: 30.5%  ichi: 32.5%  Combo: 29.4%  → Winner: ichi
  Salesforce (CRM)... ✅  MA: 34.7%  ichi: 28.7%  Combo: 40.9%  → Winner: Combo
  Adobe (ADBE)... ✅  MA: 38.1%  ichi: 30.1%  Combo: 36.0%  → Winner: MA
  Intel (INTC)... ✅  MA: 32.9%  ichi: 27.5%  Combo: 16.7%  → Winner: MA
  AMD (AMD)... ✅  MA: 38.0%  ichi: 27.9%  Combo: 28.6%  → Winner: MA
  Cisco (CSCO)... ✅  MA: 38.4%  ichi: 25.2%  Combo: 21.4%  → Winner: MA
  IBM (IBM)... ✅  MA: 39.5%  ichi: 35.8%  Combo: 42.6%  → Winner: C

In [118]:
# ════════════════════════════════════════════════════════════════
# Summary table — all assets
# ════════════════════════════════════════════════════════════════
display(combo_multi_df[[
    'Asset', 'Best MA', 'MA Win Rate %', 'MA Expectancy',
    'ichi Win Rate %', 'ichi Expectancy',
    'Combo Win Rate %', 'Combo Expectancy', 'Combo Trades', 'Combo Wins?'
]].style
    .background_gradient(cmap='RdYlGn',
        subset=['MA Win Rate %', 'ichi Win Rate %', 'Combo Win Rate %'])
    .background_gradient(cmap='RdYlGn',
        subset=['MA Expectancy', 'ichi Expectancy', 'Combo Expectancy'])
    .format({
        'MA Win Rate %':    '{:.1f}%',
        'ichi Win Rate %': '{:.1f}%',
        'Combo Win Rate %': '{:.1f}%',
        'MA Expectancy':    '{:+.2f}R',
        'ichi Expectancy': '{:+.2f}R',
        'Combo Expectancy': '{:+.2f}R',
    })
    .set_caption(f'📊 Multi-Asset: MA vs ichi vs Combo | Target {TARGET_R}R'))

,Asset,Best MA,MA Win Rate %,MA Expectancy,ichi Win Rate %,ichi Expectancy,Combo Win Rate %,Combo Expectancy,Combo Trades,Combo Wins?
0,Apple,MA20,36.5%,+0.09R,35.0%,+0.05R,25.0%,-0.25R,24,❌ No
1,Microsoft,MA50,42.4%,+0.27R,35.5%,+0.06R,31.6%,-0.05R,76,❌ No
2,Nvidia,MA100,38.6%,+0.16R,27.5%,-0.17R,32.1%,-0.04R,28,❌ No
3,Alphabet,MA100,37.9%,+0.14R,34.5%,+0.03R,52.4%,+0.57R,21,✅ Yes
4,Meta,MA50,37.2%,+0.12R,31.6%,-0.05R,16.1%,-0.52R,56,❌ No
5,Amazon,MA50,36.3%,+0.09R,32.5%,-0.03R,45.5%,+0.37R,55,✅ Yes
6,Oracle,MA200,30.5%,-0.09R,32.5%,-0.03R,29.4%,-0.12R,17,❌ No
7,Salesforce,MA100,34.7%,+0.04R,28.7%,-0.14R,40.9%,+0.23R,22,✅ Yes
8,Adobe,MA100,38.1%,+0.14R,30.1%,-0.10R,36.0%,+0.08R,25,❌ No
9,Intel,MA200,32.9%,-0.01R,27.5%,-0.17R,16.7%,-0.50R,12,❌ No


In [119]:
# ════════════════════════════════════════════════════════════════
# Verdict summary — does combo consistently win?
# ════════════════════════════════════════════════════════════════

n_assets      = len(combo_multi_df)
combo_wins    = (combo_multi_df['Combo Wins?'] == '✅ Yes').sum()
ma_wins       = (combo_multi_df['Winner'] == 'MA').sum()
ichi_wins    = (combo_multi_df['Winner'] == 'ichi').sum()
combo_is_best = (combo_multi_df['Winner'] == 'Combo').sum()

avg_ma_wr     = combo_multi_df['MA Win Rate %'].mean()
avg_ichi_wr  = combo_multi_df['ichi Win Rate %'].mean()
avg_combo_wr  = combo_multi_df['Combo Win Rate %'].mean()
avg_ma_exp    = combo_multi_df['MA Expectancy'].mean()
avg_ichi_exp = combo_multi_df['ichi Expectancy'].mean()
avg_combo_exp = combo_multi_df['Combo Expectancy'].mean()
avg_trades    = combo_multi_df['Combo Trades'].mean()

summary_rows = [
    ('Method',             'MA (best period)',  'ichimoku',        'MA + ichi Combo'),
    ('Avg Win Rate %',     f'{avg_ma_wr:.1f}%', f'{avg_ichi_wr:.1f}%', f'{avg_combo_wr:.1f}%'),
    ('Avg Expectancy',     f'{avg_ma_exp:+.2f}R', f'{avg_ichi_exp:+.2f}R', f'{avg_combo_exp:+.2f}R'),
    ('Best on N assets',   f'{ma_wins}/{n_assets}', f'{ichi_wins}/{n_assets}', f'{combo_is_best}/{n_assets}'),
    ('Beats BOTH others',  '—',                '—',                f'{combo_wins}/{n_assets}'),
    ('Avg combo trades',   '—',                '—',                f'{avg_trades:.0f}'),
]

df_summary = pd.DataFrame(summary_rows[1:], columns=summary_rows[0])

def highlight_combo(col):
    return ['background-color: rgba(255,215,0,0.15)' if col.name == 'MA + Ichimoku Combo' else '' for _ in col]

display(df_summary.style
    .apply(highlight_combo, axis=0)
    .hide(axis='index')
    .set_caption(f'📊 Multi-Asset Combo Verdict — {n_assets} assets | Target {TARGET_R}R'))

print(f"""
  Combo beats BOTH individual methods : {combo_wins}/{n_assets} assets  ({combo_wins/n_assets*100:.0f}%)
  Combo is best overall               : {combo_is_best}/{n_assets} assets
  Avg combo win rate vs MA            : {avg_combo_wr:.1f}% vs {avg_ma_wr:.1f}%
  Avg combo expectancy vs MA          : {avg_combo_exp:+.2f}R vs {avg_ma_exp:+.2f}R
""")

Method,MA (best period),ichimoku,MA + ichi Combo
Avg Win Rate %,37.0%,31.9%,33.4%
Avg Expectancy,+0.11R,-0.04R,+0.00R
Best on N assets,28/50,5/50,17/50
Beats BOTH others,—,—,17/50
Avg combo trades,—,—,34



  Combo beats BOTH individual methods : 17/50 assets  (34%)
  Combo is best overall               : 17/50 assets
  Avg combo win rate vs MA            : 33.4% vs 37.0%
  Avg combo expectancy vs MA          : +0.00R vs +0.11R



In [120]:
# ════════════════════════════════════════════════════════════════
# Visual — Grouped bar chart per asset
# ════════════════════════════════════════════════════════════════

def plot_combo_multi_asset(df):
    assets = df['Asset'].tolist()

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=('Win Rate % per Asset', 'Expectancy (R) per Asset')
    )

    for method, col, color in [
        ('MA (best)',     'MA Win Rate %',    '#2962ff'),
        ('Ichimoku',     'ichi Win Rate %', '#e91e63'),
        ('Combo MA+ichimoku','Combo Win Rate %', '#ffd700'),
    ]:
        fig.add_trace(go.Bar(
            name=method, x=assets, y=df[col],
            marker_color=color, text=df[col].apply(lambda x: f'{x:.1f}%'),
            textposition='outside'
        ), row=1, col=1)

    for method, col, color in [
        ('MA (best)',     'MA Expectancy',    '#2962ff'),
        ('Ichimoku',     'ichi Expectancy', '#e91e63'),
        ('Combo MA+ichimoku','Combo Expectancy', '#ffd700'),
    ]:
        fig.add_trace(go.Bar(
            name=method, x=assets, y=df[col],
            marker_color=color, text=df[col].apply(lambda x: f'{x:+.2f}R'),
            textposition='outside', showlegend=False
        ), row=1, col=2)

    fig.add_hline(y=50, line_color='white', line_dash='dash', row=1, col=1)
    fig.add_hline(y=0,  line_color='white', line_dash='dash', row=1, col=2)

    fig.update_layout(
        height=500, template='plotly_dark', barmode='group',
        title=dict(
            text=f'BackTestLAB — Multi-Asset: MA vs Ichimoku vs Combo | Target {TARGET_R}R',
            font=dict(size=16, color='gold')
        ),
        legend=dict(orientation='h', y=1.08)
    )
    fig.show()


plot_combo_multi_asset(combo_multi_df)

---
## ⚖️ SECTION 6 — Scientific Verdict

In [121]:
# ════════════════════════════════════════════════════════════════
# SECTION 6 — Scientific Verdict (updated)
# ════════════════════════════════════════════════════════════════

def plot_scientific_verdict(df, score_df, ichi_score_df, combo_wr,
                             best_ma_col, best_ichi_level, symbol, target_r):

    # ── Collect all results ───────────────────────────────────────
    ma_wr   = float(score_df.loc[score_df['Win Rate %'].idxmax(), 'Win Rate %'])
    ma_lbl  = str(score_df.loc[score_df['Win Rate %'].idxmax(), 'MA'])
    ma_exp  = float(score_df.loc[score_df['Win Rate %'].idxmax(), 'Expectancy'])
    ma_n    = int(score_df.loc[score_df['Win Rate %'].idxmax(), 'WIN  ✅']
                  + score_df.loc[score_df['Win Rate %'].idxmax(), 'LOSS ❌'])

    ichi_wr  = float(ichi_score_df.loc[ichi_score_df['Win Rate %'].idxmax(), 'Win Rate %'])
    ichi_lbl = str(ichi_score_df.loc[ichi_score_df['Win Rate %'].idxmax(), 'Ichimoku Level'])
    ichi_exp = float(ichi_score_df.loc[ichi_score_df['Win Rate %'].idxmax(), 'Expectancy'])
    ichi_n   = int(ichi_score_df.loc[ichi_score_df['Win Rate %'].idxmax(), 'WIN  ✅']
                   + ichi_score_df.loc[ichi_score_df['Win Rate %'].idxmax(), 'LOSS ❌'])

    combo_trades = df.attrs.get('combo_trades', [])
    combo_dec    = [t for t in combo_trades if t['outcome'] in ('win','loss')]
    combo_n      = len(combo_dec)
    combo_exp    = round(combo_wr/100*target_r - (1-combo_wr/100), 2)

    methods  = [f'{ma_lbl} (MA)', f'{ichi_lbl} (Ichimoku)', f'{ma_lbl.upper()} + {ichi_lbl} (Combo)']
    wr_vals  = [ma_wr, ichi_wr, combo_wr]
    exp_vals = [ma_exp, ichi_exp, combo_exp]
    n_vals   = [ma_n, ichi_n, combo_n]
    colors   = ['#2962ff', '#e91e63', '#ffd700']

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=(
            f'Win Rate % — {symbol}',
            'Trade Count (decided)',
        )


    )

    # Win Rate
    fig.add_trace(go.Bar(
        x=methods, y=wr_vals, marker_color=colors,
        text=[f'{v:.1f}%' for v in wr_vals],
        textposition='outside', showlegend=False
    ), row=1, col=1)
    fig.add_hline(y=50, line_color='white', line_dash='dash', row=1, col=1)

    '''
    # Expectancy
    fig.add_trace(go.Bar(
        x=methods, y=exp_vals, marker_color=colors,
        text=[f'{v:+.2f}R' for v in exp_vals],
        textposition='outside', showlegend=False
    ), row=1, col=2)
    fig.add_hline(y=0, line_color='white', line_dash='dash', row=1, col=2)
    '''

    # Trade count
    fig.add_trace(go.Bar(
        x=methods, y=n_vals, marker_color=colors,
        text=n_vals, textposition='outside', showlegend=False
    ), row=1, col=2)

    fig.update_layout(
        height=500, template='plotly_dark',
        title=dict(
            text=f'BackTestLAB — Video #10 — Scientific Verdict | Target {target_r}R',
            font=dict(size=17, color='gold')
        )
    )
    fig.show()


plot_scientific_verdict(
    df_main, score_df, ichi_score_df, combo_wr,
    best_ma_col, best_ichi_level, DEMO_LABEL, TARGET_R
)

In [35]:
# ── Print final summary ───────────────────────────────────────────
ma_wr_final   = float(score_df['Win Rate %'].max())
ichi_wr_final = float(ichi_score_df['Win Rate %'].max())
combo_dec     = [t for t in df_main.attrs.get('combo_trades',[]) if t['outcome'] in ('win','loss')]

print(f"""
╔══════════════════════════════════════════════════════════════╗
║       📊 BackTestLAB — Video #10  Scientific Verdict
║       Asset: {DEMO_LABEL:<20}  Target: {TARGET_R}R              ║
╠══════════════════════════════════════════════════════════════╣
║  METHOD             WIN RATE   EXPECTANCY                    ║
║  ─────────────────────────────────────────                   ║
║  {score_df.loc[score_df['Win Rate %'].idxmax(),'MA']:<6} (Moving Average)  {ma_wr_final:>5.1f}%
║  {best_ichi_level} (Ichimoku)      {ichi_wr_final:>5.1f}%
║  MA + ichimoku (Combo)    {combo_wr:>5.1f}%    {round(combo_wr/100*TARGET_R-(1-combo_wr/100),2):>+.2f}R  ← only {len(combo_dec)} trades
╠══════════════════════════════════════════════════════════════╣
║  KEY TAKEAWAYS                                               ║
║  1. Longer MAs (100/200) hold as S/R better than short MAs   ║
║  2. {best_ichi_level} is the most reliable Ichimoku S/R level║
║  3. Combining both → fewer trades but higher quality         ║
╚══════════════════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════════════════╗
║       📊 BackTestLAB — Video #10  Scientific Verdict        
║       Asset: EUR/USD               Target: 2.0R              ║
╠══════════════════════════════════════════════════════════════╣
║  METHOD             WIN RATE   EXPECTANCY                    ║
║  ─────────────────────────────────────────                   ║
║  MA100  (Moving Average)   31.3%  
║  senkou_b (Ichimoku)       29.9%    
║  MA + ichimoku (Combo)     32.1%    -0.04R  ← only 78 trades
╠══════════════════════════════════════════════════════════════╣
║  KEY TAKEAWAYS                                               ║
║  1. Longer MAs (100/200) hold as S/R better than short MAs   ║
║  2. senkou_b is the most reliable Ichimoku S/R level║
║  3. Combining both → fewer trades but higher quality         ║
╚══════════════════════════════════════════════════════════════╝

